# Code as Policies の蒸留：Gemini を教師にして小型モデルにロボットコードを書かせる

前の Notebook（`Code_as_Policies_Gemini_JP.ipynb`）では、Gemini に few-shot プロンプトを与えてロボット制御コードを生成させました。この Notebook では、その **Gemini 版 CaP を「教師」として教師データを作り、手元で動く小型のコード生成モデル（生徒）をファインチューニング** します。

## なぜ蒸留するのか

| | 教師（Gemini + few-shot） | 生徒（ファインチューニング済み小型モデル） |
|---|---|---|
| プロンプト | 数千トークンの few-shot 例が毎回必要 | **指示と物体リストだけ**（例は学習済み） |
| 費用 | API 料金・レート制限あり | 自分の GPU で動かせば API 料金なし |
| 速度 | ネットワーク往復＋長いプロンプト | 短いプロンプトで高速 |
| 汎用性 | 高い | 学習したタスクの範囲に強い |

## 全体の流れ

```
 Part A（CPU ランタイムで OK）            Part B（GPU）           Part C / D（GPU）
┌────────────────────────────┐   ┌──────────────────┐   ┌──────────────────────┐
│ ① タスクを自動生成            │   │ ④ 小型モデルを     │   │ ⑤ 生徒モデルで CaP を │
│   （指示文＋成功判定関数）     │   │   LoRA で          │   │   実行し、成功率を     │
│ ② Gemini 版 CaP で実行し、    │──►│   ファインチューニング│──►│   教師と比較          │
│   LMP の入出力をすべて記録     │   │   （SFT）          │   │ ⑥ 自由入力デモ        │
│ ③ 成功したエピソードだけを     │   └──────────────────┘   └──────────────────────┘
│   教師データ（JSONL）にする    │
└────────────────────────────┘
```

ポイントは **③ の「成功したものだけを残す」** ことです（リジェクションサンプリング）。シミュレータで実際に動かして、指示どおりの結果になったか（例：赤いブロックが青いボウルの上にあるか）を自動判定し、正しかったコードだけを生徒に学ばせます。

## 実行環境のおすすめ

- **Part A（データ作成）** は Gemini API の待ち時間がほとんどなので、**CPU ランタイム** で十分です。データは Google Drive に保存されます。
- **Part B〜D** は GPU ランタイムに切り替えてから実行します。

**実行のしかた**：最初の「⚙️ 実行設定」セルで `RUN_CPU_PART`（Part A）と `RUN_GPU_PART`（Part B〜D）を選び、「ランタイム → すべてのセルを実行」するだけです。OFF の部分のセルは自動でスキップされます。

1. CPU ランタイム：`RUN_CPU_PART = True` / `RUN_GPU_PART = False` → すべて実行
2. GPU ランタイムに切り替え：`RUN_CPU_PART = False` / `RUN_GPU_PART = True` → すべて実行

デモ動画は `cap_distill/videos/`、成功率などの記録は `cap_distill/report/` に保存されます（Google Drive）。
- GPU は自動判定します。**A100（40GB 以上）なら 7B モデル、T4 なら 1.5B モデル** を使います。

---
# 共通セットアップ

ここから「8. LMP の組み立て」までは、どのランタイムでも最初に実行してください。前の Notebook とほぼ同じ部品ですが、蒸留のために **3 つの変更** を加えています。

1. **バックエンドの差し替え**：LMP が「誰にコードを書かせるか」を、Gemini（教師）と小型モデル（生徒）で切り替えられるようにする
2. **入出力の記録**：LMP が呼ばれるたびに「入力（指示・文脈）と出力（コード）」を記録する → これが教師データになる
3. **安全装置**：生成コードがおかしな座標を指定してロボットが止まらなくなるのを防ぐタイムアウト、API のレート制限への対応

## 1. 設定

| 変数 | 意味 |
|---|---|
| `MODEL_NAME` | 教師に使う Gemini モデル（無料枠のあるモデルを推奨） |
| `REQUESTS_PER_MINUTE` | Gemini への 1 分あたりの最大リクエスト数。無料枠のレート制限より少し低めに設定する |
| `N_TRAIN_EPISODES` | 教師データ作成のために実行するエピソード（タスク）の数 |
| `HOLDOUT_TEMPLATES` | 学習に使わず、評価だけに使うタスクの種類（汎化性能を調べたいとき） |
| `RUN_DATA_GENERATION` | Part A のデータ生成を実行するか（GPU ランタイムでは False に） |
| `SAVE_DIR` | データとモデルの保存先（Colab では Google Drive） |

Gemini API キーの設定方法は前の Notebook と同じです（Colab のシークレットに `GEMINI_API_KEY` を登録）。

In [ ]:
#@title ⚙️ 実行設定（ここを選んでから「ランタイム → すべてのセルを実行」）
#@markdown **CPU ランタイム**：`RUN_CPU_PART` だけ ON（Gemini で教師データを作る）
#@markdown
#@markdown **GPU ランタイム**：`RUN_GPU_PART` だけ ON（学習・評価・デモ動画）。両方 ON にすると GPU ランタイムで全部続けて実行します。
RUN_CPU_PART = True  #@param {type:"boolean"}
RUN_GPU_PART = False  #@param {type:"boolean"}
#@markdown OFF にすると学習をスキップして、Drive に保存済みの LoRA アダプタを使います（評価やデモ動画だけやり直したいとき）
TRAIN_STUDENT = True  #@param {type:"boolean"}
#@markdown デモ動画を Google Drive（`cap_distill/videos`）に mp4 で保存する
SAVE_VIDEO = True  #@param {type:"boolean"}
#@markdown 成績の記録に付ける名前。学習設定を変えてやり直すときは別の名前にすると結果が混ざりません
EXPERIMENT_NAME = 'exp1'  #@param {type:"string"}

import os
# --- Gemini（教師）---
gemini_api_key = 'YOUR KEY HERE'
try:
    from google.colab import userdata
    gemini_api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_api_key = os.environ.get('GEMINI_API_KEY', gemini_api_key)

MODEL_NAME = 'gemini-3.5-flash-lite'
REQUESTS_PER_MINUTE = 8   # 無料枠のレート制限に合わせて調整（AI Studio で上限を確認）

# --- データ生成 ---
RUN_DATA_GENERATION = RUN_CPU_PART   # 互換用（Part A のデータ生成は RUN_CPU_PART で切り替え）
N_TRAIN_EPISODES = 150
HOLDOUT_TEMPLATES = []    # 例: ['side_of'] にすると「〜の左/右に置く」タスクを学習から除外

# --- 保存先（Colab なら Google Drive に保存して、ランタイムを切り替えても消えないようにする）---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/cap_distill'
except ImportError:
    SAVE_DIR = './cap_distill'
os.makedirs(SAVE_DIR, exist_ok=True)

EPISODES_PATH = f'{SAVE_DIR}/teacher_episodes.jsonl'   # 教師の全エピソード（失敗も含む）
SFT_TRAIN_PATH = f'{SAVE_DIR}/sft_train.jsonl'          # 学習用データ
SFT_VAL_PATH = f'{SAVE_DIR}/sft_val.jsonl'              # 検証用データ
ADAPTER_DIR = f'{SAVE_DIR}/student_lora'                # 学習済み LoRA アダプタ
EVAL_PATH = f'{SAVE_DIR}/eval_episodes.jsonl'           # 評価結果
REPORT_DIR = f'{SAVE_DIR}/report'                       # レポート用の成績・グラフ
VIDEO_DIR = f'{SAVE_DIR}/videos'                        # デモ動画
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)
print('保存先:', SAVE_DIR)


# --- 「すべてのセルを実行」用：フラグが False のセルを丸ごとスキップする ---
from IPython import get_ipython
from IPython.core.magic import register_cell_magic


@register_cell_magic
def run_if(line, cell):
    """セルの先頭に `%%run_if RUN_GPU_PART` と書くと、そのフラグが True のときだけ実行する。"""
    ip = get_ipython()
    if eval(line, ip.user_ns):
        ip.run_cell(cell).raise_error()   # エラーなら「すべて実行」をそこで止める
    else:
        print(f'スキップしました（{line.strip()} が False）')


try:
    import torch
    _has_gpu = torch.cuda.is_available()
except ImportError:
    _has_gpu = False
print(f'実行する部分: CPU パート(Part A)={RUN_CPU_PART} / GPU パート(Part B〜D)={RUN_GPU_PART} '
      f'/ 生徒の学習={TRAIN_STUDENT} / 実験名={EXPERIMENT_NAME}')
if RUN_GPU_PART and not _has_gpu:
    raise RuntimeError('RUN_GPU_PART = True ですが GPU がありません。'
                       '「ランタイム → ランタイムのタイプを変更」で GPU を選ぶか、RUN_GPU_PART を False にしてください。')

## 2. ライブラリのインストールとインポート

前の Notebook のライブラリに加えて、結果の集計に `pandas` を使います。学習用のライブラリ（`transformers` など）は Part B でインストールします。

In [ ]:
!pip install -q google-genai numpy scipy shapely astunparse pygments pybullet imageio imageio-ffmpeg opencv-python gdown pandas

In [ ]:
import re
import ast
import copy
import json
import time
import base64
import random
import threading
from time import sleep

import numpy as np
import pandas as pd
import cv2
import imageio
import pybullet
import pybullet_data
from IPython.display import HTML, display

import shapely
import astunparse
from shapely.geometry import *
from shapely.affinity import *

from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import TerminalFormatter

from google import genai
from google.genai import types
from google.genai import errors as genai_errors
import httpx  # google-genai が内部で使う通信ライブラリ（タイムアウト例外の判定用）

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    import matplotlib.pyplot as plt
    def cv2_imshow(img_bgr):
        plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()


# --- JSONL（1 行 1 レコードの JSON）の読み書き ---
def append_jsonl(path, record):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

## 3. Gemini 呼び出し（レート制限対応版）

前の Notebook の `gemini_complete` に、データを大量に作るための 2 つの機能を加えます。

- **スロットリング**：前回の呼び出しから `60 / REQUESTS_PER_MINUTE` 秒たつまで待つ。無料枠の 1 分あたりの上限を超えないようにする
- **タイムアウト**：通信が固まっても 60 秒で打ち切って再試行する
- **再試行の上限**：429（レート制限）や 5xx が続く場合は待ち時間を延ばしながら再試行し、それでもダメなら例外を投げる。**1 日の無料枠を使い切った** ときに無限に待ち続けないためです。データ生成ループはこの例外を受けて安全に停止し、翌日続きから再開できます

出力の後処理（コードブロック記号・指示の繰り返し・`import` 行の除去、stop トークンでの打ち切り）は前の Notebook と同じです。この後処理は生徒モデルの出力にも使います。

In [ ]:
# 応答が返ってこないまま固まらないよう、1 リクエスト 60 秒でタイムアウトさせる（単位はミリ秒）
client = genai.Client(api_key=gemini_api_key, http_options=types.HttpOptions(timeout=60_000))

CODE_COMPLETION_SYSTEM_INSTRUCTION = """You are a Python code completion engine.
The user message is a Python script made of example pairs: a comment describing a task, followed by the code that does it.
Continue the script by writing ONLY the Python code for the LAST comment.
Rules:
- Output raw Python code only. No markdown, no code fences, no explanations.
- Do not repeat the last comment line.
- Do not write any import statements; everything needed is already imported.
- Use only the functions and variables that appear in the examples.
- Stop after the code for the last comment. Do not write new comments or new tasks."""

RETRYABLE_STATUS_CODES = (429, 500, 502, 503, 504)
MAX_RETRIES = 6
_last_request_time = [0.0]


def _extract_code(text):
    """```python ... ``` で囲まれていれば中身だけを取り出す。"""
    m = re.search(r'```(?:python|py)?[ \t]*\n(.*?)(?:```|$)', text, re.S)
    return m.group(1) if m else text


def _postprocess(text, prompt, stop_tokens):
    """モデルの出力を「続きのコードだけ」に整える（教師・生徒の両方で使う）。"""
    text = _extract_code(text)
    last_prompt_line = prompt.strip().split('\n')[-1].strip()

    out_lines = []
    for line in text.split('\n'):
        stripped = line.strip()
        if not out_lines and (stripped == '' or stripped == last_prompt_line):
            continue
        if any(stripped.startswith(tok) for tok in stop_tokens):
            break
        if stripped.startswith('import ') or (stripped.startswith('from ') and ' import ' in stripped):
            continue
        out_lines.append(line)
    return '\n'.join(out_lines).strip()


def _throttle():
    """REQUESTS_PER_MINUTE を超えないように、前回の呼び出しからの間隔をあける。"""
    wait = 60.0 / REQUESTS_PER_MINUTE - (time.time() - _last_request_time[0])
    if wait > 0:
        sleep(wait)
    _last_request_time[0] = time.time()


# 思考（thinking）トークンも max_output_tokens に含まれるので、その分の余裕を足しておく
THINKING_TOKEN_HEADROOM = 1024


def _thinking_config(model):
    """思考をできるだけ減らす設定（速く・安く・出力が途中で切れないように）。"""
    if model.startswith('gemini-2.5'):
        # 2.5 系は thinking_budget で指定（Flash / Flash-Lite は 0 でオフ、Pro は最小 128）
        return types.ThinkingConfig(thinking_budget=0 if 'flash' in model else 128)
    # 3 系以降は thinking_level で指定（オフにはできないので最小の minimal にする）
    return types.ThinkingConfig(thinking_level='minimal')


def gemini_complete(prompt, stop=(), temperature=0.0, max_tokens=512, model=None,
                    system_instruction=CODE_COMPLETION_SYSTEM_INSTRUCTION):
    model = model or MODEL_NAME
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=max_tokens + THINKING_TOKEN_HEADROOM,
        thinking_config=_thinking_config(model),
    )

    for attempt in range(MAX_RETRIES):
        _throttle()
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=config)
            return _postprocess(response.text or '', prompt, list(stop))
        except genai_errors.APIError as e:
            if e.code not in RETRYABLE_STATUS_CODES or attempt == MAX_RETRIES - 1:
                raise
            wait = min(10 * 2 ** attempt, 60)
            print(f'Gemini API got err {e.code}. Retrying after {wait}s.')
            sleep(wait)
        except (httpx.TimeoutException, httpx.NetworkError) as e:
            if attempt == MAX_RETRIES - 1:
                raise
            print(f'Gemini API timeout / network error ({type(e).__name__}). Retrying after 10s.')
            sleep(10)

## 4. ロボットのモデルファイル（アセット）

In [ ]:
import os
# ランタイム再接続後は cwd が変わっていることがあるので /content に固定
if os.path.isdir('/content'):
    os.chdir('/content')

ASSETS = {
    'ur5e.zip':           ('1Cc_fDSBL6QiDvNT4dpfAEbhbALSVoWcc', 'ur5e/ur5e.urdf'),
    'robotiq_2f_85.zip':  ('1yOMEm-Zp_DL3nItG9RozPeJAmeOldekX', 'robotiq_2f_85/robotiq_2f_85.urdf'),
    'bowl.zip':           ('1GsqNLhEl9dd4Mc3BM0dX3MibOI1FVWNM', 'bowl/bowl.urdf'),
}
for zip_name, (file_id, urdf) in ASSETS.items():
    if os.path.exists(urdf):
        continue
    if not os.path.exists(zip_name) or os.path.getsize(zip_name) < 1000:
        !gdown --fuzzy {file_id} -O {zip_name}
    !unzip -q -o {zip_name}

missing = [urdf for _, urdf in ASSETS.values() if not os.path.exists(urdf)]
if missing:
    raise FileNotFoundError(
        f'アセットが見つかりません: {missing}\n'
        'gdown のダウンロードに失敗した可能性があります（Google Drive のアクセス制限など）。'
        '少し時間をおいてこのセルを再実行してください。')
print('アセット OK:', os.getcwd())

## 5. LMP の中核（バックエンド切り替え＋記録機能つき）

### 5.1 補助関数と FunctionParser

前の Notebook と同じです（`merge_dicts`・`var_exists`・`exec_safe`、生成コードから関数呼び出しを抽出する `FunctionParser`）。

In [ ]:
def merge_dicts(dicts):
    """複数の辞書を1つにまとめる（後ろの辞書が優先）。"""
    return {
        k: v
        for d in dicts
        for k, v in d.items()
    }


def var_exists(name, all_vars):
    """name という名前の変数・関数が all_vars の中に存在するか。"""
    try:
        eval(name, all_vars)
    except:
        exists = False
    else:
        exists = True
    return exists


def exec_safe(code_str, gvars=None, lvars=None):
    """LLMが生成したコードを、最低限の安全チェックをしてから実行する。"""
    banned_phrases = ['import', '__']
    for phrase in banned_phrases:
        assert phrase not in code_str, f'生成コードに禁止語 "{phrase}" が含まれています:\n{code_str}'

    if gvars is None:
        gvars = {}
    if lvars is None:
        lvars = {}
    # 生成コードの中から exec / eval を使えないようにする
    empty_fn = lambda *args, **kwargs: None
    custom_gvars = merge_dicts([
        gvars,
        {'exec': empty_fn, 'eval': empty_fn}
    ])
    exec(code_str, custom_gvars, lvars)

In [ ]:
class FunctionParser(ast.NodeTransformer):

    def __init__(self, fs, f_assigns):
        super().__init__()
        self._fs = fs                # 関数名 -> 呼び出し式 の文字列
        self._f_assigns = f_assigns  # 関数名 -> 代入文 の文字列

    def visit_Call(self, node):
        self.generic_visit(node)  # 入れ子になった呼び出しも先に処理する
        if isinstance(node.func, ast.Name):  # obj.method() ではなく func() の形だけを対象にする
            f_sig = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.func).strip()
            self._fs[f_name] = f_sig
        return node

    def visit_Assign(self, node):
        self.generic_visit(node)
        if isinstance(node.value, ast.Call):  # 右辺が関数呼び出しの代入文
            assign_str = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.value.func).strip()
            self._f_assigns[f_name] = assign_str
        return node

### 5.2 記録係（`CallRecorder`）と バックエンド

**記録係** は、LMP が呼ばれるたびに次の情報を 1 件として保存します。これが教師データの 1 サンプルの元になります。

| フィールド | 例 |
|---|---|
| `lmp` | `'tabletop_ui'`, `'parse_position'`, `'fgen'` など |
| `context` | `"objects = ['red block', 'blue bowl', ...]"` |
| `query` | `'# put the red block on the blue bowl.'` |
| `history` | それまでの実行履歴（`tabletop_ui` のみ） |
| `completion` | 生成されたコード |
| `latency` | 生成にかかった秒数 |

**バックエンド** は「コードを書く係」です。どちらも同じ引数を受け取る関数として定義し、LMP からはどちらが使われているかを区別しません。

- 教師（`gemini_backend`）：few-shot 例を含む **長いプロンプト（`full_prompt`）** を Gemini に渡す
- 生徒（Part C の `StudentBackend`）：few-shot 例を使わず、**短い入力（`context`・`query`・`history`）だけ** から書く

`VERBOSE = False` にすると、データ生成中の大量のログ表示を抑えられます。

In [ ]:
VERBOSE = True


def _log(msg):
    if VERBOSE:
        print(msg)


class CallRecorder:
    """LMP の入出力を記録する（教師データの元）。"""

    def __init__(self):
        self.calls = []

    def reset(self):
        self.calls = []

    def record(self, **kwargs):
        self.calls.append(kwargs)


RECORDER = CallRecorder()


def gemini_backend(lmp_name, full_prompt, context, query, history, stop, cfg):
    """教師バックエンド：few-shot プロンプト全体を Gemini に渡して続きを書かせる。"""
    return gemini_complete(
        full_prompt,
        stop=stop,
        temperature=cfg['temperature'],
        max_tokens=cfg['max_tokens'],
        model=cfg['model'],
    )

### 5.3 LMP / LMPFGen（バックエンド対応版）

前の Notebook の `LMP` / `LMPFGen` から、次の点だけを変えています。

- コンストラクタで `backend` を受け取り、`gemini_complete` を直接呼ぶ代わりに `self._backend(...)` を呼ぶ
- 生成のたびに `RECORDER.record(...)` で入出力を記録する
- 表示は `_log` 経由にして、`VERBOSE` で切り替えられるようにする

`LMPFGen`（未定義関数の生成）の記録は `lmp='fgen'`、`query='# define function: ...'` という形になります。生徒は関数の生成方法も一緒に学ぶことになります。

In [ ]:
class LMP:

    def __init__(self, name, cfg, lmp_fgen, fixed_vars, variable_vars, backend):
        self._name = name
        self._cfg = cfg
        self._base_prompt = self._cfg['prompt_text']
        self._stop_tokens = list(self._cfg['stop'])
        self._lmp_fgen = lmp_fgen
        self._fixed_vars = fixed_vars
        self._variable_vars = variable_vars
        self._backend = backend
        self.exec_hist = ''

    def clear_exec_hist(self):
        self.exec_hist = ''

    def build_prompt(self, query, context=''):
        if len(self._variable_vars) > 0:
            variable_vars_imports_str = f"from utils import {', '.join(self._variable_vars.keys())}"
        else:
            variable_vars_imports_str = ''
        prompt = self._base_prompt.replace('{variable_vars_imports}', variable_vars_imports_str)

        if self._cfg['maintain_session']:
            prompt += f'\n{self.exec_hist}'

        if context != '':
            prompt += f'\n{context}'

        use_query = f'{self._cfg["query_prefix"]}{query}{self._cfg["query_suffix"]}'
        prompt += f'\n{use_query}'

        return prompt, use_query

    def __call__(self, query, context='', **kwargs):
        prompt, use_query = self.build_prompt(query, context=context)
        history = self.exec_hist if self._cfg['maintain_session'] else ''

        # ① バックエンド（教師 or 生徒）にコードを書かせ、入出力を記録する
        t0 = time.time()
        code_str = self._backend(
            lmp_name=self._name, full_prompt=prompt, context=context,
            query=use_query, history=history, stop=self._stop_tokens, cfg=self._cfg,
        )
        RECORDER.record(lmp=self._name, context=context, query=use_query, history=history,
                        completion=code_str, latency=time.time() - t0)

        if self._cfg['include_context'] and context != '':
            to_exec = f'{context}\n{code_str}'
            to_log = f'{context}\n{use_query}\n{code_str}'
        else:
            to_exec = code_str
            to_log = f'{use_query}\n{to_exec}'
        _log(f'LMP {self._name} exec:\n\n{highlight(to_log, PythonLexer(), TerminalFormatter())}\n')

        # ② 未定義関数の生成
        new_fs = self._lmp_fgen.create_new_fs_from_code(code_str)
        self._variable_vars.update(new_fs)

        # ③ 実行
        gvars = merge_dicts([self._fixed_vars, self._variable_vars])
        lvars = kwargs
        if not self._cfg['debug_mode']:
            exec_safe(to_exec, gvars, lvars)

        self.exec_hist += f'\n{to_exec}'
        if self._cfg['maintain_session']:
            self._variable_vars.update(lvars)

        if self._cfg['has_return']:
            return lvars[self._cfg['return_val_name']]


class LMPFGen:

    def __init__(self, cfg, fixed_vars, variable_vars, backend):
        self._cfg = cfg
        self._stop_tokens = list(self._cfg['stop'])
        self._fixed_vars = fixed_vars
        self._variable_vars = variable_vars
        self._base_prompt = self._cfg['prompt_text']
        self._backend = backend

    def create_f_from_sig(self, f_name, f_sig, other_vars=None, fix_bugs=False, return_src=False):
        _log(f'Creating function: {f_sig}')

        use_query = f'{self._cfg["query_prefix"]}{f_sig}{self._cfg["query_suffix"]}'
        prompt = f'{self._base_prompt}\n{use_query}'

        t0 = time.time()
        f_src = self._backend(
            lmp_name='fgen', full_prompt=prompt, context='',
            query=use_query, history='', stop=self._stop_tokens, cfg=self._cfg,
        )
        RECORDER.record(lmp='fgen', context='', query=use_query, history='',
                        completion=f_src, latency=time.time() - t0)

        if other_vars is None:
            other_vars = {}
        gvars = merge_dicts([self._fixed_vars, self._variable_vars, other_vars])
        lvars = {}
        exec_safe(f_src, gvars, lvars)
        f = lvars[f_name]

        to_print = highlight(f'{use_query}\n{f_src}', PythonLexer(), TerminalFormatter())
        _log(f'LMP FGEN created:\n\n{to_print}\n')

        if return_src:
            return f, f_src
        return f

    def create_new_fs_from_code(self, code_str, other_vars=None, fix_bugs=False, return_src=False):
        fs, f_assigns = {}, {}
        f_parser = FunctionParser(fs, f_assigns)
        f_parser.visit(ast.parse(code_str))
        for f_name, f_assign in f_assigns.items():
            if f_name in fs:
                fs[f_name] = f_assign

        if other_vars is None:
            other_vars = {}

        new_fs = {}
        srcs = {}
        for f_name, f_sig in fs.items():
            all_vars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
            if not var_exists(f_name, all_vars):
                f, f_src = self.create_f_from_sig(f_name, f_sig, new_fs, fix_bugs=fix_bugs, return_src=True)

                f_def_body = astunparse.unparse(ast.parse(f_src).body[0].body)
                child_fs, child_f_srcs = self.create_new_fs_from_code(
                    f_def_body, other_vars=all_vars, fix_bugs=fix_bugs, return_src=True
                )

                if len(child_fs) > 0:
                    new_fs.update(child_fs)
                    srcs.update(child_f_srcs)
                    gvars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
                    lvars = {}
                    exec_safe(f_src, gvars, lvars)
                    f = lvars[f_name]

                new_fs[f_name], srcs[f_name] = f, f_src

        if return_src:
            return new_fs, srcs
        return new_fs

## 6. シミュレーション環境

定数・グリッパー・`PickPlaceEnv` は前の Notebook と同じです。

In [ ]:
# # Global constants: pick and place objects, colors, workspace bounds
COLORS = {
    'blue':   (78/255,  121/255, 167/255, 255/255),
    'red':    (255/255,  87/255,  89/255, 255/255),
    'green':  (89/255,  169/255,  79/255, 255/255),
    'orange': (242/255, 142/255,  43/255, 255/255),
    'yellow': (237/255, 201/255,  72/255, 255/255),
    'purple': (176/255, 122/255, 161/255, 255/255),
    'pink':   (255/255, 157/255, 167/255, 255/255),
    'cyan':   (118/255, 183/255, 178/255, 255/255),
    'brown':  (156/255, 117/255,  95/255, 255/255),
    'gray':   (186/255, 176/255, 172/255, 255/255),
}

CORNER_POS = {
  'top left corner':     (-0.3 + 0.05, -0.2 - 0.05, 0),
  'top side':            (0,           -0.2 - 0.05, 0),
  'top right corner':    (0.3 - 0.05,  -0.2 - 0.05, 0),
  'left side':           (-0.3 + 0.05, -0.5,        0),
  'middle':              (0,           -0.5,        0),
  'right side':          (0.3 - 0.05,  -0.5,        0),
  'bottom left corner':  (-0.3 + 0.05, -0.8 + 0.05, 0),
  'bottom side':         (0,           -0.8 + 0.05, 0),
  'bottom right corner': (0.3 - 0.05,  -0.8 + 0.05, 0),
}

ALL_BLOCKS = ['blue block', 'red block', 'green block', 'orange block', 'yellow block', 'purple block', 'pink block', 'cyan block', 'brown block', 'gray block']
ALL_BOWLS = ['blue bowl', 'red bowl', 'green bowl', 'orange bowl', 'yellow bowl', 'purple bowl', 'pink bowl', 'cyan bowl', 'brown bowl', 'gray bowl']

PIXEL_SIZE = 0.00267857
BOUNDS = np.float32([[-0.3, 0.3], [-0.8, -0.2], [0, 0.15]])  # X Y Z

In [ ]:
# Gripper (Robotiq 2F85) code

class Robotiq2F85:
  """Gripper handling for Robotiq 2F85."""

  def __init__(self, robot, tool):
    self.robot = robot
    self.tool = tool
    pos = [0.1339999999999999, -0.49199999999872496, 0.5]
    rot = pybullet.getQuaternionFromEuler([np.pi, 0, np.pi])
    urdf = 'robotiq_2f_85/robotiq_2f_85.urdf'
    self.body = pybullet.loadURDF(urdf, pos, rot)
    self.n_joints = pybullet.getNumJoints(self.body)
    self.activated = False

    # Connect gripper base to robot tool.
    pybullet.createConstraint(self.robot, tool, self.body, 0, jointType=pybullet.JOINT_FIXED, jointAxis=[0, 0, 0], parentFramePosition=[0, 0, 0], childFramePosition=[0, 0, -0.07], childFrameOrientation=pybullet.getQuaternionFromEuler([0, 0, np.pi / 2]))

    # Set friction coefficients for gripper fingers.
    for i in range(pybullet.getNumJoints(self.body)):
      pybullet.changeDynamics(self.body, i, lateralFriction=10.0, spinningFriction=1.0, rollingFriction=1.0, frictionAnchor=True)

    # Start thread to handle additional gripper constraints.
    self.motor_joint = 1
    self.constraints_thread = threading.Thread(target=self.step)
    self.constraints_thread.daemon = True
    self.constraints_thread.start()

  # Control joint positions by enforcing hard contraints on gripper behavior.
  # Set one joint as the open/close motor joint (other joints should mimic).
  def step(self):
    while True:
      try:
        currj = [pybullet.getJointState(self.body, i)[0] for i in range(self.n_joints)]
        indj = [6, 3, 8, 5, 10]
        targj = [currj[1], -currj[1], -currj[1], currj[1], currj[1]]
        pybullet.setJointMotorControlArray(self.body, indj, pybullet.POSITION_CONTROL, targj, positionGains=np.ones(5))
      except:
        return
      sleep(0.001)

  # Close gripper fingers.
  def activate(self):
    pybullet.setJointMotorControl2(self.body, self.motor_joint, pybullet.VELOCITY_CONTROL, targetVelocity=1, force=10)
    self.activated = True

  # Open gripper fingers.
  def release(self):
    pybullet.setJointMotorControl2(self.body, self.motor_joint, pybullet.VELOCITY_CONTROL, targetVelocity=-1, force=10)
    self.activated = False

  # If activated and object in gripper: check object contact.
  # If activated and nothing in gripper: check gripper contact.
  # If released: check proximity to surface (disabled).
  def detect_contact(self):
    obj, _, ray_frac = self.check_proximity()
    if self.activated:
      empty = self.grasp_width() < 0.01
      cbody = self.body if empty else obj
      if obj == self.body or obj == 0:
        return False
      return self.external_contact(cbody)
  #   else:
  #     return ray_frac < 0.14 or self.external_contact()

  # Return if body is in contact with something other than gripper
  def external_contact(self, body=None):
    if body is None:
      body = self.body
    pts = pybullet.getContactPoints(bodyA=body)
    pts = [pt for pt in pts if pt[2] != self.body]
    return len(pts) > 0  # pylint: disable=g-explicit-length-test

  def check_grasp(self):
    while self.moving():
      sleep(0.001)
    success = self.grasp_width() > 0.01
    return success

  def grasp_width(self):
    lpad = np.array(pybullet.getLinkState(self.body, 4)[0])
    rpad = np.array(pybullet.getLinkState(self.body, 9)[0])
    dist = np.linalg.norm(lpad - rpad) - 0.047813
    return dist

  def check_proximity(self):
    ee_pos = np.array(pybullet.getLinkState(self.robot, self.tool)[0])
    tool_pos = np.array(pybullet.getLinkState(self.body, 0)[0])
    vec = (tool_pos - ee_pos) / np.linalg.norm((tool_pos - ee_pos))
    ee_targ = ee_pos + vec
    ray_data = pybullet.rayTest(ee_pos, ee_targ)[0]
    obj, link, ray_frac = ray_data[0], ray_data[1], ray_data[2]
    return obj, link, ray_frac

In [ ]:
# Gym-style environment code

class PickPlaceEnv():

  def __init__(self, render=False, high_res=False, high_frame_rate=False):
    self.dt = 1/480
    self.sim_step = 0

    # Configure and start PyBullet.
    # python3 -m pybullet_utils.runServer
    # pybullet.connect(pybullet.SHARED_MEMORY)  # pybullet.GUI for local GUI.
    pybullet.connect(pybullet.DIRECT)  # pybullet.GUI for local GUI.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_GUI, 0)
    pybullet.setPhysicsEngineParameter(enableFileCaching=0)
    assets_path = os.path.abspath("")  # ur5e/ などがある /content
    pybullet.setAdditionalSearchPath(assets_path)
    pybullet.setAdditionalSearchPath(pybullet_data.getDataPath())
    pybullet.setTimeStep(self.dt)

    self.home_joints = (np.pi / 2, -np.pi / 2, np.pi / 2, -np.pi / 2, 3 * np.pi / 2, 0)  # Joint angles: (J0, J1, J2, J3, J4, J5).
    self.home_ee_euler = (np.pi, 0, np.pi)  # (RX, RY, RZ) rotation in Euler angles.
    self.ee_link_id = 9  # Link ID of UR5 end effector.
    self.tip_link_id = 10  # Link ID of gripper finger tips.
    self.gripper = None

    self.render = render
    self.high_res = high_res
    self.high_frame_rate = high_frame_rate

  def reset(self, object_list):
    pybullet.resetSimulation(pybullet.RESET_USE_DEFORMABLE_WORLD)
    pybullet.setGravity(0, 0, -9.8)
    self.cache_video = []

    # Temporarily disable rendering to load URDFs faster.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_RENDERING, 0)

    # Add robot.
    pybullet.loadURDF("plane.urdf", [0, 0, -0.001])
    self.robot_id = pybullet.loadURDF("ur5e/ur5e.urdf", [0, 0, 0], flags=pybullet.URDF_USE_MATERIAL_COLORS_FROM_MTL)
    self.ghost_id = pybullet.loadURDF("ur5e/ur5e.urdf", [0, 0, -10])  # For forward kinematics.
    self.joint_ids = [pybullet.getJointInfo(self.robot_id, i) for i in range(pybullet.getNumJoints(self.robot_id))]
    self.joint_ids = [j[0] for j in self.joint_ids if j[2] == pybullet.JOINT_REVOLUTE]

    # Move robot to home configuration.
    for i in range(len(self.joint_ids)):
      pybullet.resetJointState(self.robot_id, self.joint_ids[i], self.home_joints[i])

    # Add gripper.
    if self.gripper is not None:
      while self.gripper.constraints_thread.is_alive():
        self.constraints_thread_active = False
    self.gripper = Robotiq2F85(self.robot_id, self.ee_link_id)
    self.gripper.release()

    # Add workspace.
    plane_shape = pybullet.createCollisionShape(pybullet.GEOM_BOX, halfExtents=[0.3, 0.3, 0.001])
    plane_visual = pybullet.createVisualShape(pybullet.GEOM_BOX, halfExtents=[0.3, 0.3, 0.001])
    plane_id = pybullet.createMultiBody(0, plane_shape, plane_visual, basePosition=[0, -0.5, 0])
    pybullet.changeVisualShape(plane_id, -1, rgbaColor=[0.2, 0.2, 0.2, 1.0])

    # Load objects according to config.
    self.object_list = object_list
    self.obj_name_to_id = {}
    obj_xyz = np.zeros((0, 3))
    for obj_name in object_list:
      if ('block' in obj_name) or ('bowl' in obj_name):

        # Get random position 15cm+ from other objects.
        while True:
          rand_x = np.random.uniform(BOUNDS[0, 0] + 0.1, BOUNDS[0, 1] - 0.1)
          rand_y = np.random.uniform(BOUNDS[1, 0] + 0.1, BOUNDS[1, 1] - 0.1)
          rand_xyz = np.float32([rand_x, rand_y, 0.03]).reshape(1, 3)
          if len(obj_xyz) == 0:
            obj_xyz = np.concatenate((obj_xyz, rand_xyz), axis=0)
            break
          else:
            nn_dist = np.min(np.linalg.norm(obj_xyz - rand_xyz, axis=1)).squeeze()
            if nn_dist > 0.15:
              obj_xyz = np.concatenate((obj_xyz, rand_xyz), axis=0)
              break
        
        object_color = COLORS[obj_name.split(' ')[0]]
        object_type = obj_name.split(' ')[1]
        object_position = rand_xyz.squeeze()
        if object_type == 'block':
          object_shape = pybullet.createCollisionShape(pybullet.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
          object_visual = pybullet.createVisualShape(pybullet.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
          object_id = pybullet.createMultiBody(0.01, object_shape, object_visual, basePosition=object_position)
        elif object_type == 'bowl':
          object_position[2] = 0
          object_id = pybullet.loadURDF("bowl/bowl.urdf", object_position, useFixedBase=1)
        pybullet.changeVisualShape(object_id, -1, rgbaColor=object_color)
        self.obj_name_to_id[obj_name] = object_id
    

    # Re-enable rendering.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_RENDERING, 1)

    for _ in range(200):
      pybullet.stepSimulation()

    # record object positions at reset
    self.init_pos = {name: self.get_obj_pos(name) for name in object_list}

    return self.get_observation()

  def servoj(self, joints):
    """Move to target joint positions with position control."""
    pybullet.setJointMotorControlArray(
      bodyIndex=self.robot_id,
      jointIndices=self.joint_ids,
      controlMode=pybullet.POSITION_CONTROL,
      targetPositions=joints,
      positionGains=[0.01]*6)
  
  def movep(self, position):
    """Move to target end effector position."""
    joints = pybullet.calculateInverseKinematics(
        bodyUniqueId=self.robot_id,
        endEffectorLinkIndex=self.tip_link_id,
        targetPosition=position,
        targetOrientation=pybullet.getQuaternionFromEuler(self.home_ee_euler),
        maxNumIterations=100)
    self.servoj(joints)

  def get_ee_pos(self):
    ee_xyz = np.float32(pybullet.getLinkState(self.robot_id, self.tip_link_id)[0])
    return ee_xyz

  def step(self, action=None):
    """Do pick and place motion primitive."""
    pick_pos, place_pos = action['pick'].copy(), action['place'].copy()

    # Set fixed primitive z-heights.
    hover_xyz = np.float32([pick_pos[0], pick_pos[1], 0.2])
    if pick_pos.shape[-1] == 2:
      pick_xyz = np.append(pick_pos, 0.025)
    else:
      pick_xyz = pick_pos
      pick_xyz[2] = 0.025
    if place_pos.shape[-1] == 2:
      place_xyz = np.append(place_pos, 0.15)
    else:
      place_xyz = place_pos
      place_xyz[2] = 0.15

    # Move to object.
    ee_xyz = self.get_ee_pos()
    while np.linalg.norm(hover_xyz - ee_xyz) > 0.01:
      self.movep(hover_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    while np.linalg.norm(pick_xyz - ee_xyz) > 0.01:
      self.movep(pick_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    # Pick up object.
    self.gripper.activate()
    for _ in range(240):
      self.step_sim_and_render()
    while np.linalg.norm(hover_xyz - ee_xyz) > 0.01:
      self.movep(hover_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()
    
    for _ in range(50):
      self.step_sim_and_render()
    
    # Move to place location.
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    # Place down object.
    while (not self.gripper.detect_contact()) and (place_xyz[2] > 0.03):
      place_xyz[2] -= 0.001
      self.movep(place_xyz)
      for _ in range(3):
        self.step_sim_and_render()
    self.gripper.release()
    for _ in range(240):
      self.step_sim_and_render()
    place_xyz[2] = 0.2
    ee_xyz = self.get_ee_pos()
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()
    place_xyz = np.float32([0, -0.5, 0.2])
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    observation = self.get_observation()
    reward = self.get_reward()
    done = False
    info = {}
    return observation, reward, done, info

  def set_alpha_transparency(self, alpha: float) -> None:
    for id in range(20):
      visual_shape_data = pybullet.getVisualShapeData(id)
      for i in range(len(visual_shape_data)):
        object_id, link_index, _, _, _, _, _, rgba_color = visual_shape_data[i]
        rgba_color = list(rgba_color[0:3]) +  [alpha]
        pybullet.changeVisualShape(
            self.robot_id, linkIndex=i, rgbaColor=rgba_color)      
        pybullet.changeVisualShape(
            self.gripper.body, linkIndex=i, rgbaColor=rgba_color)

  def step_sim_and_render(self):
    pybullet.stepSimulation()
    self.sim_step += 1

    interval = 40 if self.high_frame_rate else 60
    # Render current image at 8 FPS.
    if self.sim_step % interval == 0 and self.render:
      self.cache_video.append(self.get_camera_image())

  def get_camera_image(self):
    if not self.high_res:
      image_size = (240, 240)
      intrinsics = (120., 0, 120., 0, 120., 120., 0, 0, 1)
    else:
      image_size=(360, 360)
      intrinsics=(180., 0, 180., 0, 180., 180., 0, 0, 1)
    color, _, _, _, _ = env.render_image(image_size, intrinsics)
    return color

  def get_reward(self):
    return None

  def get_observation(self):
    observation = {}

    # Render current image.
    color, depth, position, orientation, intrinsics = self.render_image()

    # Get heightmaps and colormaps.
    points = self.get_pointcloud(depth, intrinsics)
    position = np.float32(position).reshape(3, 1)
    rotation = pybullet.getMatrixFromQuaternion(orientation)
    rotation = np.float32(rotation).reshape(3, 3)
    transform = np.eye(4)
    transform[:3, :] = np.hstack((rotation, position))
    points = self.transform_pointcloud(points, transform)
    heightmap, colormap, xyzmap = self.get_heightmap(points, color, BOUNDS, PIXEL_SIZE)

    observation["image"] = colormap
    observation["xyzmap"] = xyzmap

    return observation

  def render_image(self, image_size=(720, 720), intrinsics=(360., 0, 360., 0, 360., 360., 0, 0, 1)):

    # Camera parameters.
    position = (0, -0.85, 0.4)
    orientation = (np.pi / 4 + np.pi / 48, np.pi, np.pi)
    orientation = pybullet.getQuaternionFromEuler(orientation)
    zrange = (0.01, 10.)
    noise=True

    # OpenGL camera settings.
    lookdir = np.float32([0, 0, 1]).reshape(3, 1)
    updir = np.float32([0, -1, 0]).reshape(3, 1)
    rotation = pybullet.getMatrixFromQuaternion(orientation)
    rotm = np.float32(rotation).reshape(3, 3)
    lookdir = (rotm @ lookdir).reshape(-1)
    updir = (rotm @ updir).reshape(-1)
    lookat = position + lookdir
    focal_len = intrinsics[0]
    znear, zfar = (0.01, 10.)
    viewm = pybullet.computeViewMatrix(position, lookat, updir)
    fovh = (image_size[0] / 2) / focal_len
    fovh = 180 * np.arctan(fovh) * 2 / np.pi

    # Notes: 1) FOV is vertical FOV 2) aspect must be float
    aspect_ratio = image_size[1] / image_size[0]
    projm = pybullet.computeProjectionMatrixFOV(fovh, aspect_ratio, znear, zfar)

    # Render with OpenGL camera settings.
    _, _, color, depth, segm = pybullet.getCameraImage(
        width=image_size[1],
        height=image_size[0],
        viewMatrix=viewm,
        projectionMatrix=projm,
        shadow=1,
        flags=pybullet.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX,
        renderer=pybullet.ER_BULLET_HARDWARE_OPENGL)

    # Get color image.
    color_image_size = (image_size[0], image_size[1], 4)
    color = np.array(color, dtype=np.uint8).reshape(color_image_size)
    color = color[:, :, :3]  # remove alpha channel
    if noise:
      color = np.int32(color)
      color += np.int32(np.random.normal(0, 3, color.shape))
      color = np.uint8(np.clip(color, 0, 255))

    # Get depth image.
    depth_image_size = (image_size[0], image_size[1])
    zbuffer = np.float32(depth).reshape(depth_image_size)
    depth = (zfar + znear - (2 * zbuffer - 1) * (zfar - znear))
    depth = (2 * znear * zfar) / depth
    if noise:
      depth += np.random.normal(0, 0.003, depth.shape)

    intrinsics = np.float32(intrinsics).reshape(3, 3)
    return color, depth, position, orientation, intrinsics

  def get_pointcloud(self, depth, intrinsics):
    """Get 3D pointcloud from perspective depth image.
    Args:
      depth: HxW float array of perspective depth in meters.
      intrinsics: 3x3 float array of camera intrinsics matrix.
    Returns:
      points: HxWx3 float array of 3D points in camera coordinates.
    """
    height, width = depth.shape
    xlin = np.linspace(0, width - 1, width)
    ylin = np.linspace(0, height - 1, height)
    px, py = np.meshgrid(xlin, ylin)
    px = (px - intrinsics[0, 2]) * (depth / intrinsics[0, 0])
    py = (py - intrinsics[1, 2]) * (depth / intrinsics[1, 1])
    points = np.float32([px, py, depth]).transpose(1, 2, 0)
    return points

  def transform_pointcloud(self, points, transform):
    """Apply rigid transformation to 3D pointcloud.
    Args:
      points: HxWx3 float array of 3D points in camera coordinates.
      transform: 4x4 float array representing a rigid transformation matrix.
    Returns:
      points: HxWx3 float array of transformed 3D points.
    """
    padding = ((0, 0), (0, 0), (0, 1))
    homogen_points = np.pad(points.copy(), padding,
                            'constant', constant_values=1)
    for i in range(3):
      points[Ellipsis, i] = np.sum(transform[i, :] * homogen_points, axis=-1)
    return points

  def get_heightmap(self, points, colors, bounds, pixel_size):
    """Get top-down (z-axis) orthographic heightmap image from 3D pointcloud.
    Args:
      points: HxWx3 float array of 3D points in world coordinates.
      colors: HxWx3 uint8 array of values in range 0-255 aligned with points.
      bounds: 3x2 float array of values (rows: X,Y,Z; columns: min,max) defining
        region in 3D space to generate heightmap in world coordinates.
      pixel_size: float defining size of each pixel in meters.
    Returns:
      heightmap: HxW float array of height (from lower z-bound) in meters.
      colormap: HxWx3 uint8 array of backprojected color aligned with heightmap.
      xyzmap: HxWx3 float array of XYZ points in world coordinates.
    """
    width = int(np.round((bounds[0, 1] - bounds[0, 0]) / pixel_size))
    height = int(np.round((bounds[1, 1] - bounds[1, 0]) / pixel_size))
    heightmap = np.zeros((height, width), dtype=np.float32)
    colormap = np.zeros((height, width, colors.shape[-1]), dtype=np.uint8)
    xyzmap = np.zeros((height, width, 3), dtype=np.float32)

    # Filter out 3D points that are outside of the predefined bounds.
    ix = (points[Ellipsis, 0] >= bounds[0, 0]) & (points[Ellipsis, 0] < bounds[0, 1])
    iy = (points[Ellipsis, 1] >= bounds[1, 0]) & (points[Ellipsis, 1] < bounds[1, 1])
    iz = (points[Ellipsis, 2] >= bounds[2, 0]) & (points[Ellipsis, 2] < bounds[2, 1])
    valid = ix & iy & iz
    points = points[valid]
    colors = colors[valid]

    # Sort 3D points by z-value, which works with array assignment to simulate
    # z-buffering for rendering the heightmap image.
    iz = np.argsort(points[:, -1])
    points, colors = points[iz], colors[iz]
    px = np.int32(np.floor((points[:, 0] - bounds[0, 0]) / pixel_size))
    py = np.int32(np.floor((points[:, 1] - bounds[1, 0]) / pixel_size))
    px = np.clip(px, 0, width - 1)
    py = np.clip(py, 0, height - 1)
    heightmap[py, px] = points[:, 2] - bounds[2, 0]
    for c in range(colors.shape[-1]):
      colormap[py, px, c] = colors[:, c]
      xyzmap[py, px, c] = points[:, c]
    colormap = colormap[::-1, :, :]  # Flip up-down.
    xv, yv = np.meshgrid(np.linspace(BOUNDS[0, 0], BOUNDS[0, 1], height),
                         np.linspace(BOUNDS[1, 0], BOUNDS[1, 1], width))
    xyzmap[:, :, 0] = xv
    xyzmap[:, :, 1] = yv
    xyzmap = xyzmap[::-1, :, :]  # Flip up-down.
    heightmap = heightmap[::-1, :]  # Flip up-down.
    return heightmap, colormap, xyzmap
  
  def on_top_of(self, obj_a, obj_b):
    """
    check if obj_a is on top of obj_b
    condition 1: l2 distance on xy plane is less than a threshold
    condition 2: obj_a is higher than obj_b
    """
    obj_a_pos = self.get_obj_pos(obj_a)
    obj_b_pos = self.get_obj_pos(obj_b)
    xy_dist = np.linalg.norm(obj_a_pos[:2] - obj_b_pos[:2])
    if obj_b in CORNER_POS:
      is_near = xy_dist < 0.06
      return is_near
    elif 'bowl' in obj_b:
      is_near = xy_dist < 0.06
      is_higher = obj_a_pos[2] > obj_b_pos[2]
      return is_near and is_higher
    else:
      is_near = xy_dist < 0.04
      is_higher = obj_a_pos[2] > obj_b_pos[2]
      return is_near and is_higher
  
  def get_obj_id(self, obj_name):
    try:
      if obj_name in self.obj_name_to_id:
        obj_id = self.obj_name_to_id[obj_name]
      else:
        obj_name = obj_name.replace('circle', 'bowl').replace('square', 'block').replace('small', '').strip()
        obj_id = self.obj_name_to_id[obj_name]
    except:
      print(f'requested_name="{obj_name}"')
      print(f'available_objects_and_id="{self.obj_name_to_id}')
    return obj_id
  
  def get_obj_pos(self, obj_name):
    obj_name = obj_name.replace('the', '').replace('_', ' ').strip()
    if obj_name in CORNER_POS:
      position = np.float32(np.array(CORNER_POS[obj_name]))
    else:
      pick_id = self.get_obj_id(obj_name)
      pose = pybullet.getBasePositionAndOrientation(pick_id)
      position = np.float32(pose[0])
    return position
  
  def get_bounding_box(self, obj_name):
    obj_id = self.get_obj_id(obj_name)
    return pybullet.getAABB(obj_id)

### 6.1 タイムアウト付きの環境（`SafePickPlaceEnv`）

`PickPlaceEnv.step` は「手先が目標位置に 1cm 以内に近づくまでループ」する作りなので、生成コードが **ロボットの届かない座標** を指定すると永遠に終わりません。人が見ているデモなら止めればよいですが、何百回も自動で回すデータ生成では致命的です。

そこで `PickPlaceEnv` を継承し、1 エピソードのシミュレーションステップ数が上限を超えたら `SimTimeout` 例外を投げるようにします。元のクラスは変更せず、`step_sim_and_render` だけを上書きしています（すべての動作がこのメソッドを通るため）。

PyBullet への接続（`pybullet.connect`）は `__init__` で行われるので、環境は **1 回だけ作って、エピソードごとに `reset` で使い回し** ます。

In [ ]:
class SimTimeout(Exception):
    pass


class SafePickPlaceEnv(PickPlaceEnv):
    MAX_SIM_STEPS_PER_EPISODE = 60000  # dt=1/480 秒なので、シミュレーション時間で約 125 秒

    def reset(self, object_list):
        obs = super().reset(object_list)
        self.episode_start_step = self.sim_step
        return obs

    def step_sim_and_render(self):
        super().step_sim_and_render()
        if self.sim_step - self.episode_start_step > self.MAX_SIM_STEPS_PER_EPISODE:
            raise SimTimeout('シミュレーションのステップ数が上限を超えました')


# 環境は 1 つだけ作って使い回す（get_camera_image がグローバル変数 env を参照するため名前は env）
env = SafePickPlaceEnv(render=False)

## 7. ロボット API・プロンプト・設定

`LMP_wrapper`、6 種類の few-shot プロンプト、LMP の設定（`cfg_tabletop`）は前の Notebook と同じです。教師はこのプロンプトを使ってコードを書きます。

In [ ]:
class LMP_wrapper():

  def __init__(self, env, cfg, render=False):
    self.env = env
    self._cfg = cfg
    self.object_names = list(self._cfg['env']['init_objs'])
    
    self._min_xy = np.array(self._cfg['env']['coords']['bottom_left'])
    self._max_xy = np.array(self._cfg['env']['coords']['top_right'])
    self._range_xy = self._max_xy - self._min_xy

    self._table_z = self._cfg['env']['coords']['table_z']
    self.render = render

  def is_obj_visible(self, obj_name):
    return obj_name in self.object_names

  def get_obj_names(self):
    return self.object_names[::]

  def denormalize_xy(self, pos_normalized):
    return pos_normalized * self._range_xy + self._min_xy

  def get_corner_positions(self):
    unit_square = box(0, 0, 1, 1)
    normalized_corners = np.array(list(unit_square.exterior.coords))[:4]
    corners = np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))
    return corners

  def get_side_positions(self):
    side_xs = np.array([0, 0.5, 0.5, 1])
    side_ys = np.array([0.5, 0, 1, 0.5])
    normalized_side_positions = np.c_[side_xs, side_ys]
    side_positions = np.array(([self.denormalize_xy(corner) for corner in normalized_side_positions]))
    return side_positions

  def get_obj_pos(self, obj_name):
    # return the xy position of the object in robot base frame
    return self.env.get_obj_pos(obj_name)[:2]

  def get_obj_position_np(self, obj_name):
    return self.get_pos(obj_name)

  def get_bbox(self, obj_name):
    # return the axis-aligned object bounding box in robot base frame (not in pixels)
    # the format is (min_x, min_y, max_x, max_y)
    bbox = self.env.get_bounding_box(obj_name)
    return bbox

  def get_color(self, obj_name):
    for color, rgb in COLORS.items():
      if color in obj_name:
        return rgb

  def pick_place(self, pick_pos, place_pos):
    pick_pos_xyz = np.r_[pick_pos, [self._table_z]]
    place_pos_xyz = np.r_[place_pos, [self._table_z]]
    pass

  def put_first_on_second(self, arg1, arg2):
    # put the object with obj_name on top of target
    # target can either be another object name, or it can be an x-y position in robot base frame
    pick_pos = self.get_obj_pos(arg1) if isinstance(arg1, str) else arg1
    place_pos = self.get_obj_pos(arg2) if isinstance(arg2, str) else arg2
    self.env.step(action={'pick': pick_pos, 'place': place_pos})

  def get_robot_pos(self):
    # return robot end-effector xy position in robot base frame
    return self.env.get_ee_pos()

  def goto_pos(self, position_xy):
    # move the robot end-effector to the desired xy position while maintaining same z
    ee_xyz = self.env.get_ee_pos()
    position_xyz = np.concatenate([position_xy, ee_xyz[-1]])
    while np.linalg.norm(position_xyz - ee_xyz) > 0.01:
      self.env.movep(position_xyz)
      self.env.step_sim_and_render()
      ee_xyz = self.env.get_ee_pos()

  def follow_traj(self, traj):
    for pos in traj:
      self.goto_pos(pos)

  def get_corner_positions(self):
    normalized_corners = np.array([
        [0, 1],
        [1, 1],
        [0, 0],
        [1, 0]
    ])
    return np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))

  def get_side_positions(self):
    normalized_sides = np.array([
        [0.5, 1],
        [1, 0.5],
        [0.5, 0],
        [0, 0.5]
    ])
    return np.array(([self.denormalize_xy(side) for side in normalized_sides]))

  def get_corner_name(self, pos):
    corner_positions = self.get_corner_positions()
    corner_idx = np.argmin(np.linalg.norm(corner_positions - pos, axis=1))
    return ['top left corner', 'top right corner', 'bottom left corner', 'bottom right corner'][corner_idx]

  def get_side_name(self, pos):
    side_positions = self.get_side_positions()
    side_idx = np.argmin(np.linalg.norm(side_positions - pos, axis=1))
    return ['top side', 'right side', 'bottom side', 'left side'][side_idx]

In [ ]:
prompt_tabletop_ui = '''
# Python 2D robot control script
import numpy as np
from env_utils import put_first_on_second, get_obj_pos, get_obj_names, say, get_corner_name, get_side_name, is_obj_visible, stack_objects_in_order
from plan_utils import parse_obj_name, parse_position, parse_question, transform_shape_pts

objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# place the yellow block on the yellow bowl.
say('Ok - putting the yellow block on the yellow bowl')
put_first_on_second('yellow block', 'yellow bowl')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# which block did you move.
say('I moved the yellow block')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# move the green block to the top right corner.
say('Got it - putting the green block on the top right corner')
corner_pos = parse_position('top right corner')
put_first_on_second('green block', corner_pos)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# stack the blue bowl on the yellow bowl on the green block.
order_bottom_to_top = ['green block', 'yellow block', 'blue bowl']
say(f'Sure - stacking from top to bottom: {", ".join(order_bottom_to_top)}')
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# move the cyan block into its corresponding bowl.
matches = {'cyan block': 'cyan bowl'}
say('Got it - placing the cyan block on the cyan bowl')
for first, second in matches.items():
  put_first_on_second(first, get_obj_pos(second))
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# make a line of blocks on the right side.
say('No problem! Making a line of blocks on the right side')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
line_pts = parse_position(f'a 30cm vertical line on the right with {len(block_names)} points')
for block_name, pt in zip(block_names, line_pts):
  put_first_on_second(block_name, pt)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# put the small banana colored thing in between the blue bowl and green block.
say('Sure thing - putting the yellow block between the blue bowl and the green block')
target_pos = parse_position('a point in the middle betweeen the blue bowl and the green block')
put_first_on_second('yellow block', target_pos)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# can you cut the bowls in half.
say('no, I can only move objects around')
objects = ['yellow block', 'green block', 'yellow bowl', 'gray block', 'gray bowl', 'green bowl']
# stack the blocks on the right side with the gray one on the bottom.
say('Ok. stacking the blocks on the right side with the gray block on the bottom')
right_side = parse_position('the right side')
put_first_on_second('gray block', right_side)
order_bottom_to_top = ['gray block', 'green block', 'yellow block']
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# hide the blue bowl.
bowl_name = np.random.choice(['yellow bowl', 'green bowl'])
say(f'Sounds good! Hiding the blue bowl under the {bowl_name}')
put_first_on_second(bowl_name, 'blue bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the grass-colored bowl to the left.
say('Sure - moving the green bowl left by 10 centimeters')
left_pos = parse_position('a point 10cm left of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# why did you move the red bowl.
say(f'I did not move the red bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# undo that.
say('Sure - moving the green bowl right by 10 centimeters')
left_pos = parse_position('a point 10cm right of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the top most block to the corner closest to the bottom most block.
top_block_name = parse_obj_name('top most block', f'objects = {get_obj_names()}')
bottom_block_name = parse_obj_name('bottom most block', f'objects = {get_obj_names()}')
closest_corner_pos = parse_position(f'the corner closest to the {bottom_block_name}', f'objects = {get_obj_names()}')
say(f'Putting the {top_block_name} on the {get_corner_name(closest_corner_pos)}')
put_first_on_second(top_block_name, closest_corner_pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the brown bowl to the side closest to the green block.
closest_side_position = parse_position('the side closest to the green block')
say(f'Got it - putting the brown bowl on the {get_side_name(closest_side_position)}')
put_first_on_second('brown bowl', closest_side_position)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the green block to the right of the bowl that has the blue block.
bowl_name = parse_obj_name('the bowl that has the blue block', f'objects = {get_obj_names()}')
if bowl_name:
  target_pos = parse_position(f'a point 10cm to the right of the {bowl_name}')
  say(f'No problem - placing the green block to the right of the {bowl_name}')
  put_first_on_second('green block', target_pos)
else:
  say('There are no bowls that has the blue block')
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the other blocks to the bottom corners.
block_names = parse_obj_name('blocks other than the blue block', f'objects = {get_obj_names()}')
corners = parse_position('the bottom corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['pink block', 'gray block', 'orange block']
# move the pinkish colored block on the bottom side.
say('Ok - putting the pink block on the bottom side')
bottom_side_pos = parse_position('the bottom side')
put_first_on_second('pink block', bottom_side_pos)
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# is the blue block to the right of the yellow bowl?
if parse_question('is the blue block to the right of the yellow bowl?', f'objects = {get_obj_names()}'):
  say('yes, there is a blue block to the right of the yellow bow')
else:
  say('no, there is\'t a blue block to the right of the yellow bow')
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# how many yellow objects are there?
n_yellow_objs = parse_question('how many yellow objects are there', f'objects = {get_obj_names()}')
say(f'there are {n_yellow_objs} yellow object')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the left most block to the green bowl.
left_block_name = parse_obj_name('left most block', f'objects = {get_obj_names()}')
say(f'Moving the {left_block_name} on the green bowl')
put_first_on_second(left_block_name, 'green bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the other blocks to different corners.
block_names = parse_obj_name(f'blocks other than the {left_block_name}', f'objects = {get_obj_names()}')
corners = parse_position('the corners')
say(f'Ok - moving the other {len(block_names)} blocks to different corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl.
if parse_question('is the pink block on the green bowl', f'objects = {get_obj_names()}'):
  say('Yes - the pink block is on the green bowl.')
else:
  say('No - the pink block is not on the green bowl.')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl.
left_block_names =  parse_question('what are the blocks left of the green bowl', f'objects = {get_obj_names()}')
if len(left_block_names) > 0:
  say(f'These blocks are left of the green bowl: {", ".join(left_block_names)}')
else:
  say('There are no blocks left of the green bowl')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# imagine that the bowls are different biomes on earth and imagine that the blocks are parts of a building.
say('ok')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# now build a tower in the grasslands.
order_bottom_to_top = ['green bowl', 'blue block', 'green block', 'yellow block']
say('stacking the blocks on the green bowl')
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['yellow block', 'green block', 'yellow bowl', 'gray block', 'gray bowl', 'green bowl']
# show me what happens when the desert gets flooded by the ocean.
say('putting the yellow bowl on the blue bowl')
put_first_on_second('yellow bowl', 'blue bowl')
objects = ['pink block', 'gray block', 'orange block']
# move all blocks 5cm toward the top.
say('Ok - moving all blocks 5cm toward the top')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name in block_names:
  target_pos = parse_position(f'a point 5cm above the {block_name}')
  put_first_on_second(block_name, target_pos)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make a triangle of blocks in the middle.
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
triangle_pts = parse_position(f'a triangle with size 10cm around the middle with {len(block_names)} points')
say('Making a triangle of blocks around the middle of the workspace')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make the triangle smaller.
triangle_pts = transform_shape_pts('scale it by 0.5x', shape_pts=triangle_pts)
say('Making the triangle smaller')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['brown bowl', 'red block', 'brown block', 'red bowl', 'pink bowl', 'pink block']
# put the red block on the farthest bowl.
farthest_bowl_name = parse_obj_name('the bowl farthest from the red block', f'objects = {get_obj_names()}')
say(f'Putting the red block on the {farthest_bowl_name}')
put_first_on_second('red block', farthest_bowl_name)
'''.strip()

prompt_parse_obj_name = '''
import numpy as np
from env_utils import get_obj_pos, parse_position
from utils import get_obj_positions_np

objects = ['blue block', 'cyan block', 'purple bowl', 'gray bowl', 'brown bowl', 'pink block', 'purple block']
# the block closest to the purple bowl.
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=get_obj_pos('purple bowl'))
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the blocks.
ret_val = ['brown block', 'blue block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the brown objects.
ret_val = ['brown bowl', 'brown block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# a fruit that's not the apple
fruit_names = ['banana', 'apple']
for fruit_name in fruit_names:
    if fruit_name != 'apple':
        ret_val = fruit_name
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# blocks above the brown bowl.
block_names = ['blue block', 'cyan block', 'purple block']
brown_bowl_pos = get_obj_pos('brown bowl')
use_block_names = []
for block_name in block_names:
    if get_obj_pos(block_name)[1] > brown_bowl_pos[1]:
        use_block_names.append(block_name)
ret_val = use_block_names
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the blue block.
ret_val = 'blue block'
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the block closest to the bottom right corner.
corner_pos = parse_position('bottom right corner')
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=corner_pos)
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the left most block.
block_names = ['green block', 'brown block', 'blue block']
block_positions = get_obj_positions_np(block_names)
left_block_idx = np.argsort(block_positions[:, 0])[0]
left_block_name = block_names[left_block_idx]
ret_val = left_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the bowl on near the top.
bowl_names = ['brown bowl', 'green bowl', 'blue bowl']
bowl_positions = get_obj_positions_np(bowl_names)
top_bowl_idx = np.argsort(bowl_positions[:, 1])[-1]
top_bowl_name = bowl_names[top_bowl_idx]
ret_val = top_bowl_name
objects = ['yellow bowl', 'purple block', 'yellow block', 'purple bowl', 'pink bowl', 'pink block']
# the third bowl from the right.
bowl_names = ['yellow bowl', 'purple bowl', 'pink bowl']
bowl_positions = get_obj_positions_np(bowl_names)
bowl_idx = np.argsort(bowl_positions[:, 0])[-3]
bowl_name = bowl_names[bowl_idx]
ret_val = bowl_name
'''.strip()

prompt_parse_position = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *
from env_utils import denormalize_xy, parse_obj_name, get_obj_names, get_obj_pos

# a 30cm horizontal line in the middle with 3 points.
middle_pos = denormalize_xy([0.5, 0.5]) 
start_pos = middle_pos + [-0.3/2, 0]
end_pos = middle_pos + [0.3/2, 0]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=3)
ret_val = points
# a 20cm vertical line near the right with 4 points.
middle_pos = denormalize_xy([1, 0.5]) 
start_pos = middle_pos + [0, -0.2/2]
end_pos = middle_pos + [0, 0.2/2]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=4)
ret_val = points
# a diagonal line from the top left to the bottom right corner with 5 points.
top_left_corner = denormalize_xy([0, 1])
bottom_right_corner = denormalize_xy([1, 0])
line = make_line(start=top_left_corner, end=bottom_right_corner)
points = interpolate_pts_on_line(line=line, n=5)
ret_val = points
# a triangle with size 10cm with 3 points.
polygon = make_triangle(size=0.1, center=denormalize_xy([0.5, 0.5]))
points = get_points_from_polygon(polygon)
ret_val = points
# the corner closest to the sun colored block.
block_name = parse_obj_name('the sun colored block', f'objects = {get_obj_names()}')
corner_positions = np.array([denormalize_xy(pos) for pos in [[0, 0], [0, 1], [1, 1], [1, 0]]])
closest_corner_pos = get_closest_point(points=corner_positions, point=get_obj_pos(block_name))
ret_val = closest_corner_pos
# the side farthest from the right most bowl.
bowl_name = parse_obj_name('the right most bowl', f'objects = {get_obj_names()}')
side_positions = np.array([denormalize_xy(pos) for pos in [[0.5, 0], [0.5, 1], [1, 0.5], [0, 0.5]]])
farthest_side_pos = get_farthest_point(points=side_positions, point=get_obj_pos(bowl_name))
ret_val = farthest_side_pos
# a point above the third block from the bottom.
block_name = parse_obj_name('the third block from the bottom', f'objects = {get_obj_names()}')
ret_val = get_obj_pos(block_name) + [0.1, 0]
# a point 10cm left of the bowls.
bowl_names = parse_obj_name('the bowls', f'objects = {get_obj_names()}')
bowl_positions = get_all_object_positions_np(obj_names=bowl_names)
left_obj_pos = bowl_positions[np.argmin(bowl_positions[:, 0])] + [-0.1, 0]
ret_val = left_obj_pos
# the bottom side.
bottom_pos = denormalize_xy([0.5, 0])
ret_val = bottom_pos
# the top corners.
top_left_pos = denormalize_xy([0, 1])
top_right_pos = denormalize_xy([1, 1])
ret_val = [top_left_pos, top_right_pos]
'''.strip()

prompt_parse_question = '''
from utils import get_obj_pos, get_obj_names, parse_obj_name, bbox_contains_pt, is_obj_visible

objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# is the blue block to the right of the yellow bowl?
ret_val = get_obj_pos('blue block')[0] > get_obj_pos('yellow bowl')[0]
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# how many yellow objects are there?
yellow_object_names = parse_obj_name('the yellow objects', f'objects = {get_obj_names()}')
ret_val = len(yellow_object_names)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl?
ret_val = bbox_contains_pt(container_name='green bowl', obj_name='pink block')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
green_bowl_pos = get_obj_pos('green bowl')
left_block_names = []
for block_name in block_names:
  if get_obj_pos(block_name)[0] < green_bowl_pos[0]:
    left_block_names.append(block_name)
ret_val = left_block_names
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# is the sun colored block above the blue bowl?
sun_block_name = parse_obj_name('sun colored block', f'objects = {get_obj_names()}')
sun_block_pos = get_obj_pos(sun_block_name)
blue_bowl_pos = get_obj_pos('blue bowl')
ret_val = sun_block_pos[1] > blue_bowl_pos[1]
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# is the green block below the blue bowl?
ret_val = get_obj_pos('green block')[1] < get_obj_pos('blue bowl')[1]
'''.strip()

prompt_transform_shape_pts = '''
import numpy as np
from utils import get_obj_pos, get_obj_names, parse_position, parse_obj_name

# make it bigger by 1.5.
new_shape_pts = scale_pts_around_centroid_np(shape_pts, scale_x=1.5, scale_y=1.5)
# move it to the right by 10cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0.1, 0])
# move it to the top by 20cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0, 0.2])
# rotate it clockwise by 40 degrees.
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=-np.deg2rad(40))
# rotate by 30 degrees and make it slightly smaller
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=np.deg2rad(30))
new_shape_pts = scale_pts_around_centroid_np(new_shape_pts, scale_x=0.7, scale_y=0.7)
# move it toward the blue block.
block_name = parse_obj_name('the blue block', f'objects = {get_obj_names()}')
block_pos = get_obj_pos(block_name)
mean_delta = np.mean(block_pos - shape_pts, axis=1)
new_shape_pts = translate_pts_np(shape_pts, mean_delta)
'''.strip()

prompt_fgen = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *

from env_utils import get_obj_pos, get_obj_names
from ctrl_utils import put_first_on_second

# define function: total = get_total(xs=numbers).
def get_total(xs):
    return np.sum(xs)

# define function: y = eval_line(x, slope, y_intercept=0).
def eval_line(x, slope, y_intercept):
    return x * slope + y_intercept

# define function: pt = get_pt_to_the_left(pt, dist).
def get_pt_to_the_left(pt, dist):
    return pt + [-dist, 0]

# define function: pt = get_pt_to_the_top(pt, dist).
def get_pt_to_the_top(pt, dist):
    return pt + [0, dist]

# define function line = make_line_by_length(length=x).
def make_line_by_length(length):
  line = LineString([[0, 0], [length, 0]])
  return line

# define function: line = make_vertical_line_by_length(length=x).
def make_vertical_line_by_length(length):
  line = make_line_by_length(length)
  vertical_line = rotate(line, 90)
  return vertical_line

# define function: pt = interpolate_line(line, t=0.5).
def interpolate_line(line, t):
  pt = line.interpolate(t, normalized=True)
  return np.array(pt.coords[0])

# example: scale a line by 2.
line = make_line_by_length(1)
new_shape = scale(line, xfact=2, yfact=2)

# example: put object1 on top of object0.
put_first_on_second('object1', 'object0')

# example: get the position of the first object.
obj_names = get_obj_names()
pos_2d = get_obj_pos(obj_names[0])
'''.strip()

In [ ]:
cfg_tabletop = {
  'lmps': {
    'tabletop_ui': {
      'prompt_text': prompt_tabletop_ui,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': True,
      'debug_mode': False,
      'include_context': True,
      'has_return': False,
      'return_val_name': 'ret_val',
    },
    'parse_obj_name': {
      'prompt_text': prompt_parse_obj_name,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_position': {
      'prompt_text': prompt_parse_position,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_question': {
      'prompt_text': prompt_parse_question,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'transform_shape_pts': {
      'prompt_text': prompt_transform_shape_pts,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'new_shape_pts',
    },
    'fgen': {
      'prompt_text': prompt_fgen,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# define function: ',
      'query_suffix': '.',
      'stop': ['# define', '# example'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
    }
  }
}

lmp_tabletop_coords = {
        'top_left':     (-0.3 + 0.05, -0.2 - 0.05),
        'top_side':     (0,           -0.2 - 0.05),
        'top_right':    (0.3 - 0.05,  -0.2 - 0.05),
        'left_side':    (-0.3 + 0.05, -0.5,      ),
        'middle':       (0,           -0.5,      ),
        'right_side':   (0.3 - 0.05,  -0.5,      ),
        'bottom_left':  (-0.3 + 0.05, -0.8 + 0.05),
        'bottom_side':  (0,           -0.8 + 0.05),
        'bottom_right': (0.3 - 0.05,  -0.8 + 0.05),
        'table_z':       0.0,
      }

## 8. LMP の組み立て（バックエンド指定つき）

前の Notebook の `setup_LMP` との違いは次の 2 点です。

- 引数 `backend` で教師／生徒を指定し、すべての LMP と LMPFGen に渡す
- ロボットの発話 `say()` を **リスト `said` にも保存** する。「ブロックはいくつ？」のような質問タスクで、答えが正しいかを判定するために使う

戻り値は `(lmp_tabletop_ui, said)` です。

In [ ]:
def setup_LMP(env, cfg_tabletop, backend):
    cfg_tabletop = copy.deepcopy(cfg_tabletop)
    cfg_tabletop['env'] = dict()
    cfg_tabletop['env']['init_objs'] = list(env.obj_name_to_id.keys())
    cfg_tabletop['env']['coords'] = lmp_tabletop_coords
    LMP_env = LMP_wrapper(env, cfg_tabletop)

    fixed_vars = {'np': np}
    fixed_vars.update({
        name: eval(name)
        for name in shapely.geometry.__all__ + shapely.affinity.__all__
    })
    variable_vars = {
        k: getattr(LMP_env, k)
        for k in [
            'get_bbox', 'get_obj_pos', 'get_color', 'is_obj_visible', 'denormalize_xy',
            'put_first_on_second', 'get_obj_names',
            'get_corner_name', 'get_side_name',
        ]
    }

    # 発話を記録する（質問タスクの正誤判定に使う）
    said = []
    def say(msg):
        said.append(str(msg))
        _log(f'robot says: {msg}')
    variable_vars['say'] = say

    lmp_fgen = LMPFGen(cfg_tabletop['lmps']['fgen'], fixed_vars, variable_vars, backend)
    variable_vars.update({
        k: LMP(k, cfg_tabletop['lmps'][k], lmp_fgen, fixed_vars, variable_vars, backend)
        for k in ['parse_obj_name', 'parse_position', 'parse_question', 'transform_shape_pts']
    })
    lmp_tabletop_ui = LMP(
        'tabletop_ui', cfg_tabletop['lmps']['tabletop_ui'], lmp_fgen, fixed_vars, variable_vars, backend
    )
    return lmp_tabletop_ui, said

---
# Part A：教師データの作成

## A-1. タスクの自動生成と成功判定

教師データを大量に作るには、**指示文** と、その指示が **達成できたかを判定する関数** のペアを自動で作る必要があります。ここでは「テンプレート」を用意し、シーンの物体に合わせて指示文と判定関数を作ります。

| テンプレート名 | 指示文の例 | 成功判定 |
|---|---|---|
| `block_on_bowl` | put the red block on the blue bowl | 赤ブロックが青ボウルの上にある |
| `color_alias` | put the sun-colored block on the blue bowl | 色の言い換えを解釈できたか（黄ブロックがボウルの上） |
| `corner` | move the red block to the top left corner | ブロックが左上の角付近にある |
| `closest_bowl` | put the red block in the bowl closest to it | 実行前に一番近かったボウルの上にある |
| `extreme_block` | put the left most block on the blue bowl | 実行前に一番左だったブロックがボウルの上にある |
| `side_of` | put the red block to the left of the blue bowl | ブロックがボウルより 4cm 以上左、かつ前後のずれが 8cm 以内 |
| `stack` | stack all the blocks on the blue bowl | すべてのブロックがボウルの真上付近にある |
| `matching` | put the blocks in the bowls with matching colors | 同じ色のボウルがあるブロックが、すべてその上にある |
| `count` | how many blocks are there? | ロボットの発話に正しい数が含まれる |

**工夫している点**

- 「一番近いボウル」「一番左のブロック」のような相対的な指示は、**実行前の配置で正解を計算** しておく
- 正解が紛らわしい配置（2 番目との差が 3cm 未満など）ではそのテンプレートを使わない（`None` を返して別のテンプレートを試す）
- 判定関数は `env` の **実行後の状態** を見るので、生成コードの書き方に関係なく「結果」で評価できる

In [ ]:
CORNERS = ['top left corner', 'top right corner', 'bottom left corner', 'bottom right corner']
COLOR_ALIASES = {
    'yellow': ['sun-colored', 'banana-colored'],
    'green':  ['grass-colored', 'forest-colored'],
    'blue':   ['ocean-colored', 'sky-colored'],
    'red':    ['strawberry-colored', 'fire-truck-colored'],
    'orange': ['carrot-colored', 'pumpkin-colored'],
    'purple': ['grape-colored', 'eggplant-colored'],
    'pink':   ['bubblegum-colored', 'flamingo-colored'],
    'cyan':   ['turquoise-colored', 'teal'],
    'brown':  ['chocolate-colored', 'coffee-colored'],
    'gray':   ['ash-colored', 'concrete-colored'],
}
NUM_WORDS = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten']


def _pos(name):
    return env.get_obj_pos(name)


def _dist_xy(a, b):
    return float(np.linalg.norm(np.asarray(a)[:2] - np.asarray(b)[:2]))


def _color(name):
    return name.split(' ')[0]


# 各テンプレートは (指示文, 判定関数) を返す。使えない配置なら None を返す。
# 判定関数は check(said) の形で、実行後の env と発話リスト said を見て True/False を返す。

def tpl_block_on_bowl(rng, blocks, bowls):
    b, w = rng.choice(blocks), rng.choice(bowls)
    fmt = rng.choice(['put the {b} on the {w}', 'place the {b} in the {w}',
                      'move the {b} into the {w}', 'pick up the {b} and drop it in the {w}'])
    return fmt.format(b=b, w=w), lambda said: env.on_top_of(b, w)


def tpl_color_alias(rng, blocks, bowls):
    b, w = rng.choice(blocks), rng.choice(bowls)
    alias = rng.choice(COLOR_ALIASES[_color(b)])
    return f'put the {alias} block on the {w}', lambda said: env.on_top_of(b, w)


def tpl_corner(rng, blocks, bowls):
    b, corner = rng.choice(blocks), rng.choice(CORNERS)
    return f'move the {b} to the {corner}', lambda said: env.on_top_of(b, corner)


def tpl_closest_bowl(rng, blocks, bowls):
    if len(bowls) < 2:
        return None
    b = rng.choice(blocks)
    ranked = sorted((_dist_xy(_pos(b), _pos(w)), w) for w in bowls)
    if ranked[1][0] - ranked[0][0] < 0.03:   # 紛らわしい配置は使わない
        return None
    target = ranked[0][1]
    return f'put the {b} in the bowl closest to it', lambda said: env.on_top_of(b, target)


def tpl_extreme_block(rng, blocks, bowls):
    if len(blocks) < 2:
        return None
    # (表現, 座標軸, 符号)：符号付きで小さい順に並べた先頭が正解
    desc, axis, sign = rng.choice([('left most', 0, 1), ('right most', 0, -1),
                                   ('top most', 1, -1), ('bottom most', 1, 1)])
    ranked = sorted((sign * _pos(b)[axis], b) for b in blocks)
    if ranked[1][0] - ranked[0][0] < 0.03:
        return None
    target, w = ranked[0][1], rng.choice(bowls)
    return f'put the {desc} block on the {w}', lambda said: env.on_top_of(target, w)


def tpl_side_of(rng, blocks, bowls):
    b, w = rng.choice(blocks), rng.choice(bowls)
    side = rng.choice(['left', 'right'])

    def check(said):
        (bx, by), (wx, wy) = _pos(b)[:2], _pos(w)[:2]
        ok_side = bx < wx - 0.04 if side == 'left' else bx > wx + 0.04
        return bool(ok_side and abs(by - wy) < 0.08)
    return f'put the {b} to the {side} of the {w}', check


def tpl_stack(rng, blocks, bowls):
    if len(blocks) < 2:
        return None
    w = rng.choice(bowls)

    def check(said):
        wp = _pos(w)
        return all(_dist_xy(_pos(b), wp) < 0.07 and _pos(b)[2] > wp[2] for b in blocks)
    return f'stack all the blocks on the {w}', check


def tpl_matching(rng, blocks, bowls):
    pairs = [(b, f'{_color(b)} bowl') for b in blocks if f'{_color(b)} bowl' in bowls]
    if not pairs:
        return None
    instruction = rng.choice(['put the blocks in the bowls with matching colors',
                              'put each block in the bowl of the same color'])
    return instruction, lambda said: all(env.on_top_of(b, w) for b, w in pairs)


def tpl_count(rng, blocks, bowls):
    kind = rng.choice(['blocks', 'bowls'])
    n = len(blocks) if kind == 'blocks' else len(bowls)

    def check(said):
        text = ' '.join(said).lower()
        return bool(re.search(rf'\b({n}|{NUM_WORDS[n]})\b', text))
    return f'how many {kind} are there?', check


TEMPLATES = {
    'block_on_bowl': tpl_block_on_bowl,
    'color_alias': tpl_color_alias,
    'corner': tpl_corner,
    'closest_bowl': tpl_closest_bowl,
    'extreme_block': tpl_extreme_block,
    'side_of': tpl_side_of,
    'stack': tpl_stack,
    'matching': tpl_matching,
    'count': tpl_count,
}
TRAIN_TEMPLATES = [t for t in TEMPLATES if t not in HOLDOUT_TEMPLATES]


def sample_task(rng, blocks, bowls, allowed_templates):
    """使えるテンプレートが見つかるまでランダムに試す。"""
    names = list(allowed_templates)
    rng.shuffle(names)
    for name in names:
        task = TEMPLATES[name](rng, blocks, bowls)
        if task is not None:
            return name, task[0], task[1]
    raise RuntimeError('どのテンプレートも使えないシーンでした')

## A-2. 1 エピソードの実行

`run_episode(seed, backend, allowed_templates)` は次の処理を行い、結果を辞書で返します。

1. `seed` から乱数を初期化し、ブロック 2〜4 個・ボウル 1〜4 個のシーンを作る（同じ seed なら同じシーンになるので、教師と生徒を **同じ条件で比較** できる）
   - **配置の事前チェック**：`env.reset` は「他の物体から 15cm 以上離れた位置」が見つかるまでランダムに位置を引き直し続けるので、物体が 7〜8 個あると置き場所がなくなって **無限ループ** になることがあります。そこで `env.reset` と同じ手順・同じ乱数で配置を先に試し、詰まる場合は物体を選び直します
2. テンプレートから指示文と判定関数を作る
3. 指定されたバックエンドで CaP を実行する（記録係をリセットしてから）
4. 例外（生成コードのエラー・タイムアウトなど）を捕まえて記録し、判定関数で成功／失敗を決める

Gemini API のエラー（1 日の上限に達したなど）だけは呼び出し元に伝えて、データ生成ループを止められるようにしています。

In [ ]:
def placement_is_feasible(n_objects, max_tries=5000):
    """env.reset と同じ手順で物体の配置を試し、置き場所が見つかるかを調べる。
    乱数の状態は元に戻すので、この後の env.reset はまったく同じ配置になる。"""
    state = np.random.get_state()
    placed = np.zeros((0, 2))
    tries = 0
    try:
        for _ in range(n_objects):
            while True:
                tries += 1
                if tries > max_tries:
                    return False
                x = np.random.uniform(BOUNDS[0, 0] + 0.1, BOUNDS[0, 1] - 0.1)
                y = np.random.uniform(BOUNDS[1, 0] + 0.1, BOUNDS[1, 1] - 0.1)
                p = np.float32([x, y]).reshape(1, 2)
                if len(placed) == 0 or np.min(np.linalg.norm(placed - p, axis=1)) > 0.15:
                    placed = np.concatenate((placed, p), axis=0)
                    break
        return True
    finally:
        np.random.set_state(state)


def run_episode(seed, backend, allowed_templates):
    rng = random.Random(seed)
    np.random.seed(seed)  # env.reset の物体配置も seed で決まるようにする

    # 置き場所が詰まらない組み合わせになるまで物体を選び直す
    while True:
        blocks = rng.sample(ALL_BLOCKS, rng.randint(2, 4))
        bowls = rng.sample(ALL_BOWLS, rng.randint(1, 4))
        if placement_is_feasible(len(blocks) + len(bowls)):
            break
    env.reset(blocks + bowls)

    template, instruction, check = sample_task(rng, blocks, bowls, allowed_templates)
    lmp_tabletop_ui, said = setup_LMP(env, cfg_tabletop, backend)

    RECORDER.reset()
    t0 = time.time()
    error = None
    try:
        lmp_tabletop_ui(instruction, f'objects = {env.object_list}')
    except genai_errors.APIError:
        raise  # API の上限などはループ側で処理する
    except SimTimeout as e:
        error = f'SimTimeout: {e}'
    except Exception as e:
        error = f'{type(e).__name__}: {e}'

    success = error is None and bool(check(said))
    return {
        'seed': seed,
        'template': template,
        'instruction': instruction,
        'objects': list(env.object_list),
        'success': success,
        'error': error,
        'said': said,
        'calls': list(RECORDER.calls),
        'llm_time': sum(c['latency'] for c in RECORDER.calls),
        'total_time': time.time() - t0,
    }

### 動作確認

まず 1 エピソードだけ教師で実行し、ログと記録内容を確認します。

In [ ]:
%%run_if RUN_CPU_PART
VERBOSE = True
ep = run_episode(seed=999999, backend=gemini_backend, allowed_templates=TRAIN_TEMPLATES)
print('指示:', ep['instruction'])
print('成功:', ep['success'], '| エラー:', ep['error'])
print('記録された LMP 呼び出し:', [c['lmp'] for c in ep['calls']])

## A-3. 教師データの生成ループ

`N_TRAIN_EPISODES` 回、教師（Gemini）でエピソードを実行し、結果を 1 件ずつ `teacher_episodes.jsonl` に追記します。

- **途中再開できる**：すでに保存済みの seed はスキップするので、止まっても再実行すれば続きから始まります
- **API の上限に達したら停止**：無料枠の 1 日の上限に達した場合はループを抜けます。翌日（日本時間の 16〜17 時ごろにリセット）に再実行してください
- **所要時間の目安**：1 エピソードあたり Gemini 呼び出しが 3〜8 回なので、`REQUESTS_PER_MINUTE = 8` なら 1 エピソード 0.5〜1 分程度です（150 エピソードで 1〜2 時間）

失敗したエピソードも保存しておき、後で「どんなタスクで教師が失敗するか」を分析できるようにしています。学習に使うのは成功したものだけです。

In [ ]:
%%run_if RUN_CPU_PART
VERBOSE = False
done_seeds = {e['seed'] for e in read_jsonl(EPISODES_PATH)}
print(f'保存済み: {len(done_seeds)} エピソード')

for seed in range(N_TRAIN_EPISODES):
    if seed in done_seeds:
        continue
    try:
        ep = run_episode(seed, gemini_backend, TRAIN_TEMPLATES)
    except genai_errors.APIError as e:
        print(f'Gemini API エラーのため停止します（無料枠の上限の可能性）: {e}')
        break
    append_jsonl(EPISODES_PATH, ep)
    mark = 'OK ' if ep['success'] else 'NG '
    print(f"[{seed:4d}] {mark} {ep['template']:<14} {ep['instruction']}  {ep['error'] or ''}")
VERBOSE = True

## A-4. 教師の成績を確認する

テンプレートごとの教師の成功率を見てみます。成功率が極端に低いテンプレートは、

- 教師（few-shot プロンプト）がそのタスクを苦手としている
- 判定関数が厳しすぎる（例：角に置いたときの許容誤差）

のどちらかなので、エラー内容を見て原因を調べると良いです。

In [ ]:
%%run_if RUN_CPU_PART
episodes = read_jsonl(EPISODES_PATH)
df_teacher = pd.DataFrame(episodes)
print(f'全 {len(df_teacher)} エピソード / 成功率 {df_teacher.success.mean():.1%}')
display(df_teacher.groupby('template').agg(
    episodes=('success', 'size'),
    success_rate=('success', 'mean'),
    avg_llm_time=('llm_time', 'mean'),
).round(2))

# 失敗例を少し表示
display(df_teacher[~df_teacher.success][['seed', 'template', 'instruction', 'error']].head(10))

# レポート用に保存
os.makedirs(REPORT_DIR, exist_ok=True)
df_teacher.groupby('template').agg(
    episodes=('success', 'size'), success_rate=('success', 'mean'), avg_llm_time=('llm_time', 'mean'),
).round(3).to_csv(f'{REPORT_DIR}/teacher_success_by_template.csv', encoding='utf-8-sig')
print('保存しました:', f'{REPORT_DIR}/teacher_success_by_template.csv')

## A-5. 教師データ（SFT 用データセット）の作成

成功したエピソードの LMP 呼び出しを 1 件ずつ学習サンプルにします。1 エピソードから `tabletop_ui`・`parse_obj_name`・`parse_position`・`fgen` など複数のサンプルが取れます。

- **重複の除去**：`fgen` の関数生成（例：`get_obj_positions_np`）は毎エピソード同じ入力になるので、(LMP 名・履歴・文脈・指示) が同じものは 1 件にまとめる
- **学習用と検証用に分割**：seed が 10 の倍数のエピソードを検証用にする（エピソード単位で分けるのは、同じシーンのサンプルが両方に入って検証が甘くなるのを防ぐため）

保存形式（1 行 1 サンプル）：
```json
{"seed": 3, "lmp": "parse_position", "history": "", "context": "objects = [...]",
 "query": "# a point 10cm left of the blue bowl.", "completion": "ret_val = get_obj_pos('blue bowl') + [-0.1, 0]"}
```

In [ ]:
%%run_if RUN_CPU_PART
def build_sft_records(episodes):
    seen, records = set(), []
    for ep in episodes:
        if not ep['success']:
            continue
        for c in ep['calls']:
            if not c['completion'].strip():
                continue
            key = (c['lmp'], c['history'], c['context'], c['query'])
            if key in seen:
                continue
            seen.add(key)
            records.append({'seed': ep['seed'], 'lmp': c['lmp'], 'history': c['history'],
                            'context': c['context'], 'query': c['query'], 'completion': c['completion']})
    return records


sft_records = build_sft_records(read_jsonl(EPISODES_PATH))
sft_train = [r for r in sft_records if r['seed'] % 10 != 0]
sft_val = [r for r in sft_records if r['seed'] % 10 == 0]

for path, recs in [(SFT_TRAIN_PATH, sft_train), (SFT_VAL_PATH, sft_val)]:
    with open(path, 'w', encoding='utf-8') as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f'学習用 {len(sft_train)} 件 / 検証用 {len(sft_val)} 件')
display(pd.DataFrame(sft_records).groupby('lmp').size().rename('samples'))

---
# Part B：生徒モデルのファインチューニング（GPU ランタイム）

> **ここから GPU ランタイムで実行します。** 「ランタイム → ランタイムのタイプを変更」で GPU（A100 または T4）を選び、最初の設定セルで `RUN_CPU_PART = False` / `RUN_GPU_PART = True` にして「すべてのセルを実行」してください。

## B-1. 生徒の入力形式を決める

教師は数千トークンの few-shot プロンプトを読んでコードを書いていました。生徒には **few-shot 例を見せず**、次のような短い入力からコードを書くよう学習させます。few-shot 例が教える「書き方」は、学習によってモデルの重みに覚えさせるわけです。

```
[system] You are a code-writing policy for a tabletop robot. ...
[user]   # LMP: parse_position
         objects = ['red block', 'blue bowl', ...]
         # a point 10cm left of the blue bowl.
[assistant] ret_val = get_obj_pos('blue bowl') + [-0.1, 0]
```

先頭の `# LMP: 名前` で、どの役割（tabletop_ui / parse_position / fgen など）として書くかを伝えます。1 つのモデルがすべての LMP の役割を担当します。

In [ ]:
%%run_if RUN_GPU_PART
STUDENT_SYSTEM_PROMPT = (
    'You are a code-writing policy for a tabletop robot. '
    'Given the LMP name, the context and an instruction comment, '
    'write only the Python code that should follow. No explanations.'
)


def build_student_messages(lmp, context, query, history=''):
    parts = [f'# LMP: {lmp}']
    if history.strip():
        parts.append(history.strip())
    if context.strip():
        parts.append(context.strip())
    parts.append(query)
    return [
        {'role': 'system', 'content': STUDENT_SYSTEM_PROMPT},
        {'role': 'user', 'content': '\n'.join(parts)},
    ]


# 例を表示
_sample = read_jsonl(SFT_TRAIN_PATH)[0]
for m in build_student_messages(_sample['lmp'], _sample['context'], _sample['query'], _sample['history']):
    print(f"[{m['role']}]\n{m['content']}\n")
print(f"[assistant（正解）]\n{_sample['completion']}")

## B-2. 学習ライブラリと生徒モデルの選択

| ライブラリ | 用途 |
|---|---|
| `transformers` | モデルの読み込み・学習ループ（`Trainer`） |
| `peft` | **LoRA**：モデル本体の重みは固定し、小さな追加行列だけを学習する手法。メモリと時間を大幅に節約できる |
| `accelerate` | GPU での学習の補助 |

生徒モデルはコード生成に強い **Qwen2.5-Coder（Instruct 版）** を使い、GPU のメモリ量で大きさを自動で選びます。

| GPU | モデル | 読み込み精度 | 備考 |
|---|---|---|---|
| A100（40GB 以上） | `Qwen2.5-Coder-7B-Instruct` | bfloat16 | 精度重視。学習は数十分程度 |
| T4（16GB） | `Qwen2.5-Coder-1.5B-Instruct` | float32（学習は fp16 混合精度） | T4 は bfloat16 非対応のため |

別のモデルを試したい場合は `STUDENT_BASE_MODEL` を書き換えてください（チャットテンプレートを持つ Instruct 系モデルなら同じコードで動くはずです）。

In [ ]:
%%run_if RUN_GPU_PART
# Colab に入っている古い torchao (0.10) は新しい peft と非互換なので削除 (LoRA には不要)
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate

In [ ]:
%%run_if RUN_GPU_PART
import torch
import transformers
from packaging import version
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

assert torch.cuda.is_available(), 'GPU ランタイムに切り替えてください'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_BF16 = torch.cuda.is_bf16_supported()

if gpu_mem_gb > 30:   # A100 / H100 など
    STUDENT_BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
    BATCH_SIZE, GRAD_ACCUM = 4, 4
else:                 # T4 など
    STUDENT_BASE_MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
    BATCH_SIZE, GRAD_ACCUM = 2, 8

LOAD_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print(f'GPU: {gpu_name} ({gpu_mem_gb:.0f}GB) / bf16: {USE_BF16}')
print(f'生徒モデル: {STUDENT_BASE_MODEL} / 実効バッチサイズ: {BATCH_SIZE * GRAD_ACCUM}')


def load_base_model(name):
    # transformers 4.56 以降は torch_dtype ではなく dtype という引数名
    dtype_kw = 'dtype' if version.parse(transformers.__version__) >= version.parse('4.56.0') else 'torch_dtype'
    return AutoModelForCausalLM.from_pretrained(name, device_map={'': 0}, **{dtype_kw: LOAD_DTYPE})


tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## B-3. データを「トークン列＋正解ラベル」に変換する

言語モデルの学習では、入力のトークン列に対して「次のトークン」を予測させます。ここで大事なのは **損失を計算する範囲** です。

- system・user（指示）の部分：モデルに覚えさせる必要はないので、ラベルを `-100`（損失計算から除外）にする
- assistant（コード）の部分：ここだけを正解として学習する

```
トークン : <system ...> <user # LMP: ... # put the red block ...> <assistant> say(...) put_first_on_second(...) <|im_end|>
ラベル   :   -100 ...      -100 ...                                 -100        say(...) put_first_on_second(...) <|im_end|>
```

末尾の終了トークン（`<|im_end|>`）も学習させることで、生徒は「コードを書き終えたら止まる」ことを覚えます。

文字列をチャットテンプレートで整形してからトークン化し、プロンプト部分と回答部分を **別々にトークン化してつなげる** ことで、境界がずれないようにしています。長すぎるサンプル（`MAX_SEQ_LEN` 超）は除外します。

In [ ]:
%%run_if RUN_GPU_PART and TRAIN_STUDENT
MAX_SEQ_LEN = 2048


def encode_record(rec):
    msgs = build_student_messages(rec['lmp'], rec['context'], rec['query'], rec['history'])
    prompt_text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    full_text = tokenizer.apply_chat_template(
        msgs + [{'role': 'assistant', 'content': rec['completion']}], tokenize=False)
    assert full_text.startswith(prompt_text)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(full_text[len(prompt_text):], add_special_tokens=False)['input_ids']
    input_ids = prompt_ids + answer_ids
    if len(input_ids) > MAX_SEQ_LEN:
        return None
    labels = [-100] * len(prompt_ids) + answer_ids
    return {'input_ids': input_ids, 'labels': labels}


class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        encoded = [encode_record(r) for r in records]
        self.items = [e for e in encoded if e is not None]
        print(f'{len(self.items)} 件（長すぎて除外: {len(encoded) - len(self.items)} 件）')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    """バッチ内で一番長い系列に合わせてパディングする。"""
    max_len = max(len(x['input_ids']) for x in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x['input_ids'])
        input_ids[i, :n] = torch.tensor(x['input_ids'])
        labels[i, :n] = torch.tensor(x['labels'])
        attention_mask[i, :n] = 1
    return {'input_ids': input_ids, 'labels': labels, 'attention_mask': attention_mask}


train_ds = SFTDataset(read_jsonl(SFT_TRAIN_PATH))
val_ds = SFTDataset(read_jsonl(SFT_VAL_PATH))

## B-4. LoRA の設定とモデルの準備

LoRA は、Transformer の各線形層（`q_proj`, `k_proj`, ... `down_proj`）の横に、ランク `r` の小さな行列 2 つ（A: d×r, B: r×d）を足して、その部分だけを学習します。7B モデルでも学習するパラメータは全体の 1% 未満です。

| パラメータ | 値 | 意味 |
|---|---|---|
| `r` | 16 | 追加行列のランク（大きいほど表現力が上がるがメモリも増える） |
| `lora_alpha` | 32 | LoRA の出力にかける倍率（`alpha / r` 倍） |
| `lora_dropout` | 0.05 | 過学習を防ぐドロップアウト |

メモリ節約のため **gradient checkpointing**（順伝播の途中結果を保存せず、逆伝播時に再計算する）も有効にします。

In [ ]:
%%run_if RUN_GPU_PART and TRAIN_STUDENT
model = load_base_model(STUDENT_BASE_MODEL)
model.config.use_cache = False          # 学習中は KV キャッシュ不要
model.gradient_checkpointing_enable()
model.enable_input_require_grads()      # gradient checkpointing と LoRA を併用するために必要

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## B-5. 学習

`transformers` の `Trainer` で学習します。データが数百件程度と少ないので、エポック数は 3 にしています。学習後に検証データの損失（`eval_loss`）を表示します。学習データの損失だけが下がって検証損失が上がる場合は過学習なので、エポック数を減らすかデータを増やしてください。

In [ ]:
%%run_if RUN_GPU_PART and TRAIN_STUDENT
NUM_EPOCHS = 3

training_args = TrainingArguments(
    output_dir=f'{SAVE_DIR}/trainer_out',
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    bf16=USE_BF16,
    fp16=not USE_BF16,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate,
)
_t0 = time.time()
train_result = trainer.train()
train_minutes = (time.time() - _t0) / 60

eval_metrics = trainer.evaluate(eval_dataset=val_ds) if len(val_ds) > 0 else {}
print(eval_metrics)

# LoRA アダプタだけを保存（数十〜数百 MB）
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('保存しました:', ADAPTER_DIR)

# --- レポート用に学習の記録を保存 ---
import matplotlib.pyplot as plt
training_info = {
    'experiment': EXPERIMENT_NAME,
    'student_base_model': STUDENT_BASE_MODEL,
    'gpu': gpu_name,
    'num_epochs': NUM_EPOCHS,
    'effective_batch_size': BATCH_SIZE * GRAD_ACCUM,
    'learning_rate': training_args.learning_rate,
    'lora_r': lora_config.r,
    'train_samples': len(train_ds),
    'val_samples': len(val_ds),
    'train_minutes': round(train_minutes, 1),
    'final_train_loss': train_result.training_loss,
    'eval_loss': eval_metrics.get('eval_loss'),
    'log_history': trainer.state.log_history,
}
with open(f'{REPORT_DIR}/training_{EXPERIMENT_NAME}.json', 'w', encoding='utf-8') as f:
    json.dump(training_info, f, ensure_ascii=False, indent=2)

_logs = [h for h in trainer.state.log_history if 'loss' in h]
if _logs:
    plt.figure(figsize=(6, 3.5))
    plt.plot([h['step'] for h in _logs], [h['loss'] for h in _logs], marker='o', ms=3)
    plt.xlabel('step'); plt.ylabel('train loss'); plt.title(f'Training loss ({EXPERIMENT_NAME})')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/training_loss_{EXPERIMENT_NAME}.png', dpi=150)
    plt.show()
print(f'学習時間 {train_minutes:.1f} 分 / 記録を保存しました:', REPORT_DIR)

---
# Part C：生徒モデルで CaP を動かして評価する

## C-1. 生徒バックエンド

教師の `gemini_backend` と **同じ引数** を受け取る関数（呼び出し可能なクラス）を作ります。これを `setup_LMP` に渡すだけで、CaP のすべての LMP が生徒モデルで動くようになります。

- `full_prompt`（few-shot 例を含む長いプロンプト）は **使わない**。`context`・`query`・`history` だけから学習時と同じ形式の入力を作る
- 貪欲法（`do_sample=False`）で決定的に生成する
- 出力には教師と同じ後処理（`_postprocess`）をかける

学習済みのアダプタを保存先から読み込むセルも用意しています（ランタイムを再起動した後に Part C から始める場合に使います）。

In [ ]:
%%run_if RUN_GPU_PART
# ランタイム再起動後など、学習済みモデルがメモリにない場合は保存先から読み込む
if 'model' not in globals() or not isinstance(model, PeftModel):
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
    model = PeftModel.from_pretrained(load_base_model(STUDENT_BASE_MODEL), ADAPTER_DIR)
model.eval()


class StudentBackend:
    """生徒バックエンド：ファインチューニングした小型モデルで、短い入力からコードを書く。"""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    @torch.no_grad()
    def __call__(self, lmp_name, full_prompt, context, query, history, stop, cfg):
        msgs = build_student_messages(lmp_name, context, query, history)
        text = self.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inputs = self.tokenizer(text, return_tensors='pt', add_special_tokens=False).to(self.model.device)
        out = self.model.generate(
            **inputs,
            max_new_tokens=cfg['max_tokens'],
            do_sample=False,
            use_cache=True,
            max_time=60,                    # 1 回の生成は最大 60 秒で打ち切る
            pad_token_id=self.tokenizer.pad_token_id,
        )
        completion = self.tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return _postprocess(completion, query, stop)


student_backend = StudentBackend(model, tokenizer)

### 検証データで教師と生徒の出力を見比べる

まずはシミュレーションを使わずに、検証データの数件について「教師が書いたコード」と「生徒が書いたコード」を並べてみます。完全一致しなくても、同じ意味のコードになっていれば OK です。

In [ ]:
%%run_if RUN_GPU_PART
_dummy_cfg = {'max_tokens': 512}
for rec in read_jsonl(SFT_VAL_PATH)[:5]:
    stop = cfg_tabletop['lmps'][rec['lmp']]['stop']
    student_code = student_backend(rec['lmp'], '', rec['context'], rec['query'], rec['history'], stop, _dummy_cfg)
    print('=' * 70)
    print(f"[{rec['lmp']}] {rec['query']}")
    print('--- 教師 ---')
    print(rec['completion'])
    print('--- 生徒 ---')
    print(student_code)

## C-2. シミュレーションでの成功率評価

学習に使っていない seed（100000 番台）のシーンで、生徒モデルに CaP を実行させ、テンプレートごとの成功率を測ります。評価では `HOLDOUT_TEMPLATES` も含めた **すべてのテンプレート** を使うので、学習から除外したタスクへの汎化も確認できます。

`RUN_TEACHER_EVAL = True` にすると、同じ seed で教師（Gemini）も実行して、まったく同じ条件で比較できます（API を使います）。False の場合は、Part A の教師の成功率と比べてください。

`llm_time` は 1 エピソードでコード生成にかかった合計秒数です。生徒は few-shot 例を読まない分、速くなっているはずです。

In [ ]:
%%run_if RUN_GPU_PART
N_EVAL_EPISODES = 30
RUN_TEACHER_EVAL = False
EVAL_SEEDS = range(100000, 100000 + N_EVAL_EPISODES)

agents = [('student', student_backend)]
if RUN_TEACHER_EVAL:
    agents.append(('teacher', gemini_backend))

VERBOSE = False
done = {(e['agent'], e['seed']) for e in read_jsonl(EVAL_PATH) if e.get('run', 'exp1') == EXPERIMENT_NAME}
for agent_name, backend in agents:
    for seed in EVAL_SEEDS:
        if (agent_name, seed) in done:
            continue
        ep = run_episode(seed, backend, list(TEMPLATES))
        ep['agent'] = agent_name
        ep['run'] = EXPERIMENT_NAME
        append_jsonl(EVAL_PATH, ep)
        mark = 'OK ' if ep['success'] else 'NG '
        print(f"[{agent_name}:{seed}] {mark} {ep['template']:<14} {ep['instruction']}  {ep['error'] or ''}")
VERBOSE = True

df_eval = pd.DataFrame([e for e in read_jsonl(EVAL_PATH) if e.get('run', 'exp1') == EXPERIMENT_NAME])
eval_summary = df_eval.groupby('agent').agg(
    episodes=('success', 'size'), success_rate=('success', 'mean'),
    avg_llm_time=('llm_time', 'mean'), avg_total_time=('total_time', 'mean')).round(3)
eval_by_template = df_eval.pivot_table(index='template', columns='agent', values='success', aggfunc='mean').round(3)
display(eval_summary)
display(eval_by_template)

# --- レポート用に保存（CSV は Excel でそのまま開ける）---
import matplotlib.pyplot as plt
eval_summary.to_csv(f'{REPORT_DIR}/eval_summary_{EXPERIMENT_NAME}.csv', encoding='utf-8-sig')
eval_by_template.to_csv(f'{REPORT_DIR}/eval_by_template_{EXPERIMENT_NAME}.csv', encoding='utf-8-sig')
df_eval[['agent', 'seed', 'template', 'instruction', 'success', 'error', 'llm_time', 'total_time']].to_csv(
    f'{REPORT_DIR}/eval_episodes_{EXPERIMENT_NAME}.csv', index=False, encoding='utf-8-sig')

ax = eval_by_template.plot.bar(figsize=(8, 3.5), ylim=(0, 1.05), rot=30)
ax.set_ylabel('success rate'); ax.set_title(f'Success rate by template ({EXPERIMENT_NAME})')
ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/eval_by_template_{EXPERIMENT_NAME}.png', dpi=150)
plt.show()
print('保存しました:', REPORT_DIR)

---
# Part D：生徒モデルでインタラクティブデモ

前の Notebook と同じように、自由な指示で生徒モデルを動かしてみます。学習したテンプレートに近い指示ほどうまく動き、まったく違う種類の指示（例：「三角形に並べて」）は苦手なはずです。どこまで汎化するか試してみてください。

In [ ]:
%%run_if RUN_GPU_PART
def show_video(frames, fps=25, path='demo.mp4', width=360):
    """画像のリストを mp4 に変換して Notebook 上に表示する（path に保存される）。"""
    imageio.mimsave(path, frames, fps=fps)
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls autoplay loop muted>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    ))


DEMO_LOG_PATH = f'{REPORT_DIR}/demo_runs.jsonl'   # デモ実行の記録（指示・生成コード・動画の場所）

In [ ]:
#@title 環境の初期化 { vertical-output: true }
num_blocks = 3 #@param {type:"slider", min:1, max:4, step:1}
num_bowls = 3 #@param {type:"slider", min:1, max:4, step:1}

if RUN_GPU_PART:
    env.render = True
    obj_list = (np.random.choice(ALL_BLOCKS, size=num_blocks, replace=False).tolist()
                + np.random.choice(ALL_BOWLS, size=num_bowls, replace=False).tolist())
    env.reset(obj_list)
    student_lmp, student_said = setup_LMP(env, cfg_tabletop, student_backend)

    cv2_imshow(cv2.cvtColor(env.get_camera_image(), cv2.COLOR_BGR2RGB))
    print('available objects:')
    print(obj_list)

In [ ]:
#@title 生徒モデルで実行 { vertical-output: true }
user_input = 'put the left most block on the bowl closest to it' #@param {allow-input: true, type:"string"}
keep_history = False #@param {type:"boolean"}
#@markdown 動画を保存するかどうかは、最初の設定セルの `SAVE_VIDEO` で切り替えます

if RUN_GPU_PART:
    # 繰り返し実行すると、前回までに実行したコードが exec_hist にたまり続け、
    # 生徒モデルへの入力がどんどん長くなる（学習時に見たことのない長さになり、生成が遅く・不安定になる）。
    # ふつうは毎回クリアして、1 回の指示ごとに独立して実行する。
    if not keep_history:
        student_lmp.clear_exec_hist()

    env.cache_video = []
    env.episode_start_step = env.sim_step  # タイムアウトのカウントをリセット
    RECORDER.reset()
    print('Running student policy and recording video...')
    _t0 = time.time()
    demo_error = None
    try:
        student_lmp(user_input, f'objects = {env.object_list}')
    except SimTimeout as e:
        demo_error = f'SimTimeout: {e}'
        print('タイムアウト:', e)
    demo_time = time.time() - _t0
    print(f'実行時間: {demo_time:.1f} 秒 / 履歴の長さ: {len(student_lmp.exec_hist)} 文字')

    video_path = None
    if env.cache_video:
        if SAVE_VIDEO:
            slug = re.sub(r'[^A-Za-z0-9]+', '_', user_input).strip('_')[:40]
            video_path = f"{VIDEO_DIR}/{EXPERIMENT_NAME}_{time.strftime('%Y%m%d_%H%M%S')}_{slug}.mp4"
        show_video(env.cache_video, fps=25, path=video_path or 'demo.mp4')
        if video_path:
            print('動画を保存しました:', video_path)

    # レポート用に記録（成否は自動判定できないので、動画を見て書き込んでください）
    append_jsonl(DEMO_LOG_PATH, {
        'experiment': EXPERIMENT_NAME,
        'time': time.strftime('%Y-%m-%d %H:%M:%S'),
        'instruction': user_input,
        'objects': list(env.object_list),
        'said': list(student_said),
        'error': demo_error,
        'run_seconds': round(demo_time, 1),
        'llm_seconds': round(sum(c['latency'] for c in RECORDER.calls), 1),
        'code': [{'lmp': c['lmp'], 'completion': c['completion']} for c in RECORDER.calls],
        'video': video_path,
    })

---
# Part E：レポート用のまとめ

ここまでの結果（教師データ・学習・評価・デモ）を 1 つの Markdown ファイルにまとめて、`cap_distill/report/summary_<実験名>.md` に保存します。CPU / GPU どちらのランタイムでも、そこまでに保存されている結果だけを集計します。

| 保存されるファイル（`cap_distill/report/`） | 中身 |
|---|---|
| `teacher_success_by_template.csv` | 教師（Gemini）のテンプレート別成功率 |
| `training_<実験名>.json` / `training_loss_<実験名>.png` | 学習の設定・時間・損失、損失のグラフ |
| `eval_summary_<実験名>.csv` / `eval_by_template_<実験名>.csv/.png` | 生徒（と教師）の成功率・テンプレート別成功率 |
| `eval_episodes_<実験名>.csv` | 評価の全エピソード（指示・成否・エラー） |
| `demo_runs.jsonl` | デモの指示・生成コード・動画ファイルの場所 |
| `summary_<実験名>.md` | 上をまとめた表 |

In [ ]:
from IPython.display import Markdown

def _table(df):
    try:
        return df.to_markdown()
    except ImportError:   # tabulate が無い環境
        return '```\n' + df.to_string() + '\n```'

report = [f'# 実験結果まとめ（{EXPERIMENT_NAME}）', '', f'作成日時: {time.strftime("%Y-%m-%d %H:%M:%S")}', '']

# --- Part A：教師 ---
_teacher = read_jsonl(EPISODES_PATH)
if _teacher:
    _dt = pd.DataFrame(_teacher)
    report += ['## 教師（Gemini）のデータ作成', '',
               f'- エピソード数: {len(_dt)} / 成功率: {_dt.success.mean():.1%} / 平均 LLM 時間: {_dt.llm_time.mean():.1f} 秒',
               f'- 学習データ: {len(read_jsonl(SFT_TRAIN_PATH))} 件 / 検証データ: {len(read_jsonl(SFT_VAL_PATH))} 件', '',
               _table(_dt.groupby('template').agg(episodes=('success', 'size'), success_rate=('success', 'mean')).round(2)), '']

# --- Part B：学習 ---
_train_path = f'{REPORT_DIR}/training_{EXPERIMENT_NAME}.json'
if os.path.exists(_train_path):
    with open(_train_path, encoding='utf-8') as f:
        _ti = json.load(f)
    report += ['## 生徒モデルの学習', '']
    report += [f'- {k}: {v}' for k, v in _ti.items() if k != 'log_history']
    report += ['', f'![loss](training_loss_{EXPERIMENT_NAME}.png)', '']

# --- Part C：評価 ---
_ev = [e for e in read_jsonl(EVAL_PATH) if e.get('run', 'exp1') == EXPERIMENT_NAME]
if _ev:
    _de = pd.DataFrame(_ev)
    report += ['## シミュレーションでの評価', '',
               _table(_de.groupby('agent').agg(episodes=('success', 'size'), success_rate=('success', 'mean'),
                                               avg_llm_time=('llm_time', 'mean')).round(3)), '',
               '### テンプレート別の成功率', '',
               _table(_de.pivot_table(index='template', columns='agent', values='success', aggfunc='mean').round(2)), '',
               f'![by template](eval_by_template_{EXPERIMENT_NAME}.png)', '']
    _fail = _de[~_de.success]
    if len(_fail):
        report += ['### 失敗したエピソード', '',
                   _table(_fail[['agent', 'seed', 'template', 'instruction', 'error']].set_index('seed')), '']

# --- Part D：デモ ---
_demos = [d for d in read_jsonl(f'{REPORT_DIR}/demo_runs.jsonl') if d.get('experiment') == EXPERIMENT_NAME]
if _demos:
    report += ['## デモ実行', '',
               _table(pd.DataFrame(_demos)[['time', 'instruction', 'error', 'run_seconds', 'video']]), '']

_summary_path = f'{REPORT_DIR}/summary_{EXPERIMENT_NAME}.md'
with open(_summary_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report))
display(Markdown('\n'.join(report)))
print('保存しました:', _summary_path)

---
## まとめと発展

この Notebook では、

1. テンプレートからタスクと成功判定を自動生成し、
2. Gemini 版 CaP（教師）をシミュレータで実行して、**成功したコードだけ** を集め、
3. 小型のコードモデル（生徒）を LoRA でファインチューニングし、
4. few-shot 例なしで CaP を動かせるかを成功率で評価しました。

**発展のアイデア**

- **データを増やす・多様にする**：テンプレートや言い換えを増やす。Gemini に指示文の言い換えを作らせるのも有効です（その場合も判定関数は元のテンプレートのものを使う）
- **教師の温度を上げて複数回サンプリング**：`cfg_tabletop` の `temperature` を 0.3〜0.7 にして同じタスクを複数回解かせ、成功したものを全部使うと、データの多様性が増えます
- **失敗データの活用**：失敗したエピソードを「悪い例」として、DPO などの選好学習に使う
- **ホールドアウト実験**：`HOLDOUT_TEMPLATES = ['side_of']` のようにして、見たことのない種類のタスクにどこまで汎化するかを調べる
- **生徒からの自己改善**：生徒が成功したエピソードも学習データに加えて再学習する（Expert Iteration）